In [86]:
import os
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Literal
from uuid import uuid4


In [87]:
@dataclass
class LayerSpec:
    """Schema definition for one markdown-backed index + detail layer."""

    layer_name: str          # e.g. "elements", "events"
    uuid_prefix: str         # e.g. "elt_", "evt_"
    index_fields: list[str]  # ordered pipe-delimited column names (must include "uuid")
    detail_sections: list[str]  # ordered section headings for the detail template
    index_file: str          # relative path from storage root, e.g. "story/elements.md"
    detail_dir: str          # relative path from storage root, e.g. "story/elements"
    index_preamble: str      # everything before '## Entries' in the index file

    # Maps index field names to the labels used in the detail file's
    # ## Identification section. Order matters for rendering.
    # e.g. {"kind": "Type", "display_name": "Canonical name", ...}
    index_to_detail_label: dict[str, str] = field(default_factory=dict)

    def detail_filename(self, uuid: str) -> str:
        return f"{uuid}.md"

    def detail_path(self, storage_root: Path, uuid: str) -> Path:
        return storage_root / self.detail_dir / self.detail_filename(uuid)

    def index_path(self, storage_root: Path) -> Path:
        return storage_root / self.index_file

In [88]:
class BaseLayerAccessor:
    """
    Minimal consistency engine for one markdown-backed index + detail layer.

    Responsibilities:
    - Read index + detail files into in-memory records
    - Apply approved record changes (create / update / delete) to disk
    - Keep detail file headers in sync with index-owned fields
    - Validate and repair consistency (missing detail files, header drift)
    """

    def __init__(self, spec: LayerSpec, storage_root: str | Path) -> None:
        self.spec = spec
        self.root = Path(storage_root)
        # {uuid: {field_name: value}} for index-owned fields
        self._index: dict[str, dict[str, str]] = {}
        # {uuid: {section_key: content_str}} for detail-owned sections
        self._details: dict[str, dict[str, str]] = {}
        self._load()

    # ── Read API ─────────────────────────────────────────────────

    def list_uuids(self) -> list[str]:
        """All UUIDs currently in the index, in file order."""
        return list(self._index.keys())

    def get(self, uuid: str) -> dict:
        """Merged record (index fields + detail sections) for one UUID."""
        if uuid not in self._index:
            raise KeyError(f"{self.spec.layer_name}: UUID {uuid!r} not found.")
        merged = dict(self._index[uuid])
        if uuid in self._details:
            merged.update(self._details[uuid])
        return merged

    def list_all(self) -> list[dict]:
        """All merged records in index order."""
        return [self.get(uuid) for uuid in self._index]

    def get_index_text(self) -> str:
        """Rendered index markdown (for feeding to an LLM as context)."""
        return self._render_index()

    def get_detail_text(self, uuid: str) -> str:
        """Rendered detail file markdown for one UUID."""
        if uuid not in self._index:
            raise KeyError(f"{self.spec.layer_name}: UUID {uuid!r} not found.")
        return self._render_detail(uuid)

    # ── Write API (apply approved changes) ───────────────────────

    def apply_create(self, record: dict) -> str:
        """
        Add a new record. Auto-generates UUID.
        Writes updated index file and creates the detail file.
        Returns the new UUID.
        """
        uuid = self._make_uuid()
        index_fields = {"uuid": uuid}
        detail_sections = {}

        for f in self.spec.index_fields:
            if f == "uuid":
                continue
            val = record.get(f, "")
            index_fields[f] = self._clean_index_value(f, val)

        for s in self.spec.detail_sections:
            key = self._section_key(s)
            detail_sections[key] = record.get(key, "- TBD")

        self._index[uuid] = index_fields
        self._details[uuid] = detail_sections

        self._write_index()
        self._write_detail(uuid)
        return uuid

    def apply_update(self, uuid: str, changes: dict) -> None:
        """
        Update fields on an existing record and write to disk.
        changes can mix index-owned and detail-owned fields freely.
        """
        if uuid not in self._index:
            raise KeyError(f"{self.spec.layer_name}: UUID {uuid!r} not found.")

        index_dirty = False
        detail_dirty = False
        index_field_set = set(self.spec.index_fields)
        detail_key_set = {self._section_key(s) for s in self.spec.detail_sections}

        for key, value in changes.items():
            if key == "uuid":
                continue  # never allow UUID mutation
            if key in index_field_set:
                self._index[uuid][key] = self._clean_index_value(key, value)
                index_dirty = True
            elif key in detail_key_set:
                if uuid not in self._details:
                    self._details[uuid] = {}
                self._details[uuid][key] = value
                detail_dirty = True
            else:
                raise ValueError(
                    f"{self.spec.layer_name}: Unknown field {key!r}. "
                    f"Valid index fields: {self.spec.index_fields}. "
                    f"Valid detail sections: {list(detail_key_set)}."
                )

        if index_dirty:
            self._write_index()
            # index-owned fields are mirrored in detail header, so detail needs rewrite too
            detail_dirty = True
        if detail_dirty:
            self._write_detail(uuid)

    def apply_delete(self, uuid: str) -> None:
        """Remove record from index + delete detail file."""
        if uuid not in self._index:
            raise KeyError(f"{self.spec.layer_name}: UUID {uuid!r} not found.")

        del self._index[uuid]
        self._details.pop(uuid, None)

        self._write_index()
        detail_path = self.spec.detail_path(self.root, uuid)
        if detail_path.exists():
            detail_path.unlink()

    # ── Consistency ───────────────────────────────────────────────

    def validate(self) -> list[str]:
        """
        Check consistency. Returns list of issue descriptions (empty = clean).
        - Every index UUID has a detail file
        - Detail file headers match index-owned fields
        - No orphan detail files without index entries
        """
        issues = []
        detail_dir = self.root / self.spec.detail_dir

        # Check every index entry has a detail file
        for uuid in self._index:
            dp = self.spec.detail_path(self.root, uuid)
            if not dp.exists():
                issues.append(f"Missing detail file for {uuid}: {dp}")
            else:
                # Check header consistency
                header_issues = self._check_detail_header(uuid)
                issues.extend(header_issues)

        # Check for orphan detail files
        if detail_dir.exists():
            for f in detail_dir.iterdir():
                if f.suffix == ".md":
                    file_uuid = f.stem
                    if file_uuid not in self._index:
                        issues.append(f"Orphan detail file (no index entry): {f}")

        return issues

    def repair(self) -> list[str]:
        """
        Fix consistency issues. Returns list of actions taken.
        - Missing detail files → create from template
        - Mismatched headers → overwrite with index truth
        - Orphan detail files → delete
        """
        actions = []
        detail_dir = self.root / self.spec.detail_dir

        for uuid in self._index:
            dp = self.spec.detail_path(self.root, uuid)
            if not dp.exists():
                # Create detail file from template with default sections
                if uuid not in self._details:
                    self._details[uuid] = {
                        self._section_key(s): "- TBD"
                        for s in self.spec.detail_sections
                    }
                self._write_detail(uuid)
                actions.append(f"Created detail file for {uuid}")
            else:
                header_issues = self._check_detail_header(uuid)
                if header_issues:
                    self._write_detail(uuid)
                    actions.append(f"Rewrote detail header for {uuid} (was out of sync)")

        # Remove orphan detail files
        if detail_dir.exists():
            for f in detail_dir.iterdir():
                if f.suffix == ".md" and f.stem not in self._index:
                    f.unlink()
                    actions.append(f"Deleted orphan detail file: {f.name}")

        return actions

    def reload(self) -> None:
        """Re-read everything from disk."""
        self._index.clear()
        self._details.clear()
        self._load()

    # ── Internal: loading ─────────────────────────────────────────

    def _load(self) -> None:
        """Parse index file + all detail files from disk."""
        idx_path = self.spec.index_path(self.root)
        detail_dir = self.root / self.spec.detail_dir

        # Ensure the index file and detail directory exist
        idx_path.parent.mkdir(parents=True, exist_ok=True)
        detail_dir.mkdir(parents=True, exist_ok=True)

        if not idx_path.exists():
            # Write a fresh empty index
            idx_path.write_text(self.spec.index_preamble, encoding="utf-8")

        # Parse index
        text = idx_path.read_text(encoding="utf-8")
        self._index = self._parse_index(text)

        # Parse all detail files that exist for indexed UUIDs
        for uuid in self._index:
            dp = self.spec.detail_path(self.root, uuid)
            if dp.exists():
                self._details[uuid] = self._parse_detail(dp.read_text(encoding="utf-8"))

    def _parse_index(self, text: str) -> dict[str, dict[str, str]]:
        """Parse index markdown into {uuid: {field: value}} ordered dict."""
        marker = "## Entries"
        if marker not in text:
            raise ValueError(f"{self.spec.index_file} is missing '## Entries' section.")

        _, entries_block = text.split(marker, maxsplit=1)
        records: dict[str, dict[str, str]] = {}
        fields = self.spec.index_fields

        for raw_line in entries_block.splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            # Strip leading "- " if present (the index uses list-item format)
            if line.startswith("- "):
                line = line[2:]

            parts = [p.strip() for p in line.split("|")]
            if len(parts) != len(fields):
                raise ValueError(
                    f"Index line has {len(parts)} fields, expected {len(fields)}: {raw_line!r}"
                )

            record = {f: v for f, v in zip(fields, parts)}
            uuid = record["uuid"]

            if uuid in records:
                raise ValueError(f"Duplicate UUID in {self.spec.index_file}: {uuid}")
            records[uuid] = record

        return records

    def _parse_detail(self, text: str) -> dict[str, str]:
        """
        Parse a detail file markdown into {section_key: content} for
        detail-owned sections only (ignores the ## Identification header).
        """
        sections: dict[str, str] = {}
        current_section: str | None = None
        current_lines: list[str] = []

        for line in text.splitlines():
            heading_match = re.match(r"^##\s+(.+)$", line)
            if heading_match:
                # Save previous section
                if current_section is not None:
                    sections[current_section] = "\n".join(current_lines).strip()

                heading = heading_match.group(1).strip()
                key = self._section_key(heading)
                detail_keys = {self._section_key(s) for s in self.spec.detail_sections}

                if key in detail_keys:
                    current_section = key
                    current_lines = []
                else:
                    # This is either ## Identification or unknown, skip content
                    current_section = None
                    current_lines = []
                continue

            if current_section is not None:
                current_lines.append(line)

        # Flush last section
        if current_section is not None:
            sections[current_section] = "\n".join(current_lines).strip()

        return sections

    # ── Internal: rendering ───────────────────────────────────────

    def _render_index(self) -> str:
        """Render the full index markdown from in-memory records."""
        lines = []
        for uuid, rec in self._index.items():
            vals = [rec.get(f, "") for f in self.spec.index_fields]
            lines.append("- " + " | ".join(vals))

        entries = "\n".join(lines)
        if entries:
            entries += "\n"
        return self.spec.index_preamble + entries

    def _render_detail(self, uuid: str) -> str:
        """
        Render a full detail file from index-owned header + detail-owned sections.
        Index-owned fields always come from self._index (source of truth).
        """
        rec = self._index[uuid]
        detail = self._details.get(uuid, {})

        # Title line: use display_name if present, else uuid
        title = rec.get("display_name", rec.get("summary", uuid))
        parts = [f"# {title}", "", "## Identification"]

        # Render identification fields from index, using the label mapping
        parts.append(f"- UUID: {uuid}")
        for idx_field, label in self.spec.index_to_detail_label.items():
            if idx_field == "uuid":
                continue
            val = rec.get(idx_field, "-") or "-"
            parts.append(f"- {label}: {val}")

        # Render detail-owned sections
        for section_name in self.spec.detail_sections:
            key = self._section_key(section_name)
            content = detail.get(key, "- TBD")
            parts.append("")
            parts.append(f"## {section_name}")
            parts.append(content)

        parts.append("")  # trailing newline
        return "\n".join(parts)

    # ── Internal: consistency checks ──────────────────────────────

    def _check_detail_header(self, uuid: str) -> list[str]:
        """
        Compare the ## Identification header in the on-disk detail file
        against the current index-owned fields. Return list of issues.
        """
        issues = []
        dp = self.spec.detail_path(self.root, uuid)
        if not dp.exists():
            return [f"Detail file missing for {uuid}"]

        text = dp.read_text(encoding="utf-8")
        rec = self._index[uuid]

        for idx_field, label in self.spec.index_to_detail_label.items():
            # Normalize: empty string and "-" are both "empty"
            raw_expected = rec.get(idx_field, "-") or "-"
            # Look for "- Label: value" in the file (value may be empty)
            pattern = rf"^- {re.escape(label)}:\s*(.*)$"
            match = re.search(pattern, text, re.MULTILINE)
            if match:
                raw_actual = match.group(1).strip() or "-"
                if raw_actual != raw_expected:
                    issues.append(
                        f"{uuid}: detail header '{label}' is '{raw_actual}', "
                        f"index says '{raw_expected}'"
                    )
            else:
                issues.append(f"{uuid}: detail file missing '{label}' in header")

        return issues

    # ── Internal: write helpers ───────────────────────────────────

    def _write_index(self) -> None:
        """Write the index file to disk."""
        path = self.spec.index_path(self.root)
        path.write_text(self._render_index(), encoding="utf-8")

    def _write_detail(self, uuid: str) -> None:
        """Write one detail file to disk."""
        path = self.spec.detail_path(self.root, uuid)
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(self._render_detail(uuid), encoding="utf-8")

    # ── Internal: utilities ───────────────────────────────────────

    def _make_uuid(self) -> str:
        """Generate a unique prefixed UUID for this layer."""
        while True:
            candidate = f"{self.spec.uuid_prefix}{uuid4().hex[:12]}"
            if candidate not in self._index:
                return candidate

    @staticmethod
    def _clean_index_value(field_name: str, value: str) -> str:
        """Sanitize a value for use in a pipe-delimited index line."""
        cleaned = value.strip() if value else ""
        if "|" in cleaned:
            raise ValueError(f"{field_name!r} cannot contain '|' (pipe-delimited format).")
        if "\n" in cleaned or "\r" in cleaned:
            raise ValueError(f"{field_name!r} must be single-line.")
        return cleaned

    @staticmethod
    def _section_key(section_name: str) -> str:
        """Convert a display section name to a dict key. e.g. 'Core Understanding' -> 'core_understanding'"""
        return re.sub(r"[^a-z0-9]+", "_", section_name.lower()).strip("_")

In [89]:
ELEMENTS_PREAMBLE = """# Elements

Purpose:
This file is the canonical index of stable, story-relevant elements in the world model.

Definitions:
- Element:
  A stable, story-relevant unit of the world model. An element is any distinct thing the story can meaningfully refer to, track, or reason about across time: a person, place, item, relationship, group, concept, rule, institution, creature, or other persistent entity. An element is not just any word that appears in the prose. It should be something with enough identity and continuity that the system benefits from treating it as a canonical object in the world.

- Kind:
  The broad category of what the element is. Its purpose is to place the element into the right conceptual bucket so the system can reason about it appropriately. Kind should be chosen at the level that is useful and stable, such as person, place, item, animal, relationship, concept, group, or other. It is a typing aid, not a full explanation of the element.

- Display Name:
  The canonical surface name used to represent the element in the index. Its purpose is readability and consistency. It is the main human-facing name by which the element is referred to in the world model. It should be the clearest stable name for the element, even if the manuscript sometimes refers to it in other ways.

- UUID:
  A stable canonical identifier for the element. Its purpose is identity, not meaning. It allows the system to refer to the same element reliably across the index, the detailed element file, future updates, and agent workflows, even if the display name, aliases, or understanding of the element evolves. UUIDs are generated programmatically and treated as the permanent handle for that element.

- Aliases:
  Alternative names, labels, titles, spellings, or surface references that may refer to the same element. Their purpose is recognition and matching. They help the system understand that multiple phrasings in the manuscript may point to one canonical element. Aliases should capture meaningful alternate references, not every descriptive phrase the prose happens to use.

- Identification Keys:
  Short recognition cues that help distinguish this element from others. Their purpose is practical identification, especially when names are ambiguous, absent, evolving, or reused. These should be compact, concrete signals grounded in the manuscript, such as roles, relationships, defining objects, repeated traits, or highly specific associations. They are not meant to be full descriptions or interpretations. They are quick anchors that help the system recognize who or what this element is.

Format:
- kind | display_name | uuid | aliases | identification_keys

Notes:
- one line per canonical element
- uuid is generated programmatically
- aliases are comma-separated
- identification_keys are semicolon-separated short recognition cues
- detailed element files live in ./story/elements/<uuid>.md

## Entries
"""


class ElementsAccessor(BaseLayerAccessor):
    """Concrete accessor for the elements layer."""

    def __init__(self, storage_root: str | Path) -> None:
        spec = LayerSpec(
            layer_name="elements",
            uuid_prefix="elt_",
            index_fields=["kind", "display_name", "uuid", "aliases", "identification_keys"],
            detail_sections=[
                "Core Understanding",
                "Stable Profile",
                "Interpretation",
                "Knowledge / Beliefs / Uncertainties",
                "Element-Centered Chronology",
                "Open Threads",
            ],
            index_file="story/elements.md",
            detail_dir="story/elements",
            index_preamble=ELEMENTS_PREAMBLE,
            index_to_detail_label={
                "uuid": "UUID",
                "kind": "Type",
                "display_name": "Canonical name",
                "aliases": "Aliases",
                "identification_keys": "Identification keys",
            },
        )
        super().__init__(spec, storage_root)

In [90]:
EVENTS_PREAMBLE = """# Events

Purpose:
This file is the canonical index of meaningful story events.

Definitions:
- Event:
  A bounded, world-relevant unit of change. An event is something that happens, is discovered, is decided, or is revealed in a way that meaningfully changes the story world, a character's knowledge, a relationship, a goal, a risk, or the consequences that follow. It is not merely a scene fragment or a sentence-level action. It should be stable enough to remain the same event even if later chapters refine its timing, participants, cause, or full meaning.

- UUID:
  A stable canonical identifier for the event. Its purpose is identity, not meaning. It allows the system to refer to the same event reliably across the index, the detailed event file, later updates, and agent workflows, even if the event's wording, timing, or interpretation evolves. UUIDs are generated programmatically and treated as the permanent handle for each event.

- When:
  The best currently known placement of the event in story time. Its purpose is to situate the event chronologically without forcing false precision. It may be exact, approximate, relative, inferred, or unknown, depending on what the manuscript currently supports. This field should remain open to later refinement as future chapters reveal stronger temporal grounding. It captures story-time placement, not document-edit time.

- Chapters:
  The manuscript locations where the event is evidenced, introduced, developed, or clarified. Its purpose is grounding and traceability. It tells the system and the reader where this event comes from in the text. It is an internal reference to the relevant chapter or chunk, not part of the event's meaning itself. Multiple chapters may be listed when an event is introduced in one place and clarified or extended later.

- Summary:
  A short canonical description of what the event is. Its purpose is to make the event legible at index level without retelling the full scene. It should describe the core happening in clear, neutral, world-level language, focusing on what changed or what became known. It should not drift into interpretation, prose style, emotional commentary, or speculation beyond what the manuscript supports.

Format:
- uuid | when | chapters | summary

Notes:
- one line per meaningful story event
- uuid is generated programmatically
- when may remain open if the manuscript does not yet support a stronger placement
- chapters may be a single chapter, multiple chapters, or a range
- summary should stay compact and canonical
- detailed event files live in ./story/events/<uuid>.md

## Entries
"""


class EventsAccessor(BaseLayerAccessor):
    """Concrete accessor for the events layer."""

    def __init__(self, storage_root: str | Path) -> None:
        spec = LayerSpec(
            layer_name="events",
            uuid_prefix="evt_",
            index_fields=["uuid", "when", "chapters", "summary"],
            detail_sections=[
                "Core Understanding",
                "Causal Context",
                "Consequences & Ripple Effects",
                "Participants & Roles",
                "Evidence & Grounding",
                "Open Threads",
            ],
            index_file="story/events.md",
            detail_dir="story/events",
            index_preamble=EVENTS_PREAMBLE,
            index_to_detail_label={
                "uuid": "UUID",
                "when": "When",
                "chapters": "Chapters",
                "summary": "Summary",
            },
        )
        super().__init__(spec, storage_root)

In [91]:
# ── Instantiate Elements Accessor ──
elements = ElementsAccessor(".")

print(f"Loaded {len(elements.list_uuids())} element(s)")
print()

issues = elements.validate()
if issues:
    print("⚠️  Consistency issues found:")
    for issue in issues:
        print(f"  - {issue}")
    print()
    print("Running repair ...")
    actions = elements.repair()
    for a in actions:
        print(f"  ✓ {a}")
else:
    print("✅ Elements layer is consistent.")

Loaded 0 element(s)

✅ Elements layer is consistent.


In [92]:
# ── Instantiate Events Accessor ──
events = EventsAccessor(".")

print(f"Loaded {len(events.list_uuids())} event(s)")
print()

issues = events.validate()
if issues:
    print("⚠️  Consistency issues found:")
    for issue in issues:
        print(f"  - {issue}")
    print()
    print("Running repair ...")
    actions = events.repair()
    for a in actions:
        print(f"  ✓ {a}")
else:
    print("✅ Events layer is consistent.")

Loaded 7 event(s)

✅ Events layer is consistent.


---

## Event Agent

The following cells implement the **Event Agent**: an LLM-powered agent that reads a manuscript diff and the current `events.md`, then proposes canonical create / update / delete changes to the events index.

**Flow:**
1. **Call** the agent → it returns a structured `EventAgentOutput`
2. **Review** the proposal (printed for you)
3. **Approve** → changes are applied via `EventsAccessor`
4. **Reject** → provide feedback, agent retries with full context

In [73]:
import json
import re
from abc import ABC, abstractmethod
from datetime import datetime
from difflib import unified_diff
from enum import Enum
from typing import Literal, Optional

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field, model_validator

load_dotenv()

llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=8000,
    max_retries=2,
    timeout=120,
)

print("✅ Imports loaded, LLM configured")

✅ Imports loaded, LLM configured


In [29]:
# ── Base Agent Harness ────────────────────────────────────────────

DEFAULT_FEEDBACK_TEMPLATE = """
═══ HISTORY ═══

Attempt {attempt_number}:

Your previous proposal:
<previous_output>
{previous_output}
</previous_output>

Reviewer feedback:
<feedback>
{reviewer_feedback}
</feedback>

Please revise your proposal based on the feedback above."""


class BaseAgentHarness(ABC):
    """
    Reusable LLM agent harness with structured output and
    human-in-the-loop review flow.

    Subclasses implement:
      - build_user_prompt(**context) -> str
      - format_proposal(proposal) -> str
      - apply_proposal(proposal, **kwargs) -> list[str]
    """

    def __init__(
        self,
        llm,
        system_prompt: str,
        output_schema: type[BaseModel],
        feedback_template: str | None = None,
    ):
        self.llm = llm
        self.system_prompt = system_prompt
        self.output_schema = output_schema
        self.feedback_template = feedback_template or DEFAULT_FEEDBACK_TEMPLATE
        # Review flow state
        self._proposal = None
        self._history: list[str] = []
        self._attempt: int = 0

    # ── Abstract (subclass must implement) ────────────────────

    @abstractmethod
    def build_user_prompt(self, **context) -> str:
        """Build the user message from agent-specific context."""
        ...

    @abstractmethod
    def format_proposal(self, proposal) -> str:
        """Pretty-print the proposal for human review."""
        ...

    @abstractmethod
    def apply_proposal(self, proposal, **kwargs) -> list[str]:
        """Apply the approved proposal. Returns list of action descriptions."""
        ...

    # ── Concrete (shared across all agents) ───────────────────

    def call(self, fresh: bool = False, **context):
        """Call the LLM, store proposal, return it."""
        if fresh:
            self.reset()
        self._attempt += 1
        user_content = self.build_user_prompt(**context)

        messages = [{"role": "system", "content": self.system_prompt}]
        for block in self._history:
            messages.append({"role": "user", "content": block})
        messages.append({"role": "user", "content": user_content})

        structured = self.llm.with_structured_output(self.output_schema)
        self._proposal = structured.invoke(messages)
        return self._proposal

    def review(self) -> str:
        """Format current proposal for human review."""
        if self._proposal is None:
            return "⚠️  No proposal to review. Run call() first."
        return self.format_proposal(self._proposal)

    def approve(self, **kwargs) -> list[str]:
        """Apply the current proposal and reset state."""
        if self._proposal is None:
            raise ValueError("No proposal to approve. Run call() first.")
        actions = self.apply_proposal(self._proposal, **kwargs)
        self._last_proposal = self._proposal
        self.reset()
        return actions

    def reject(self, feedback: str):
        """Record feedback for the next call."""
        if self._proposal is None:
            raise ValueError("No proposal to reject. Run call() first.")
        entry = self.feedback_template.format(
            attempt_number=self._attempt,
            previous_output=self._proposal.model_dump_json(indent=2),
            reviewer_feedback=feedback.strip(),
        )
        self._history.append(entry)
        self._proposal = None

    def reset(self):
        """Clear all review flow state."""
        self._proposal = None
        self._history = []
        self._attempt = 0

    @property
    def has_proposal(self) -> bool:
        return self._proposal is not None

    @property
    def attempt(self) -> int:
        return self._attempt


print("✅ BaseAgentHarness defined")

✅ BaseAgentHarness defined


In [30]:
# ── Shared markdown helpers ───────────────────────────────────────

def extract_section(text: str, start_heading: str, end_heading: Optional[str] = None) -> str:
    """Extract text between two markdown headings."""
    start = text.find(start_heading)
    if start == -1:
        return ""
    start += len(start_heading)
    if end_heading:
        end = text.find(end_heading, start)
        if end == -1:
            end = len(text)
    else:
        end = len(text)
    return text[start:end].strip()


def parse_bullet_lines(section_text: str) -> list[str]:
    """Extract non-TBD bullet items from a section."""
    items = []
    for raw in section_text.splitlines():
        line = raw.strip()
        if line.startswith("- "):
            value = line[2:].strip()
            if value and value != "TBD":
                items.append(value)
    return items


def parse_bullet_field(line: str, prefix: str) -> str:
    if line.startswith(prefix):
        return line[len(prefix):].strip()
    return ""


def normalize_line(x: str) -> str:
    return re.sub(r"\s+", " ", x.strip())


def merge_section_lines(
    current_lines: list[str],
    to_add: list[str],
    to_remove: list[str],
) -> list[str]:
    """Merge add/remove into an existing bullet list, deduplicating."""
    result = list(current_lines)
    remove_set = {normalize_line(x) for x in to_remove if x.strip()}
    result = [x for x in result if normalize_line(x) not in remove_set]
    existing_set = {normalize_line(x) for x in result}
    for item in to_add:
        norm = normalize_line(item)
        if norm and norm not in existing_set:
            result.append(item.strip())
            existing_set.add(norm)
    return result


def bullet_section(items: list[str]) -> list[str]:
    """Render a list as markdown bullets, or '- TBD' if empty."""
    if not items:
        return ["- TBD"]
    return [f"- {x}" for x in items]


def build_unified_diff(old_text: str, new_text: str, file_path: str) -> str:
    if old_text == new_text:
        return ""
    diff = unified_diff(
        old_text.splitlines(keepends=True),
        new_text.splitlines(keepends=True),
        fromfile=f"a/{file_path}",
        tofile=f"b/{file_path}",
    )
    return "".join(diff)


print("✅ Shared helpers defined")

✅ Shared helpers defined


In [31]:
# ── Base Detail Page Harness ──────────────────────────────────────


class DetailTarget(BaseModel):
    """A single detail file queued for update."""
    uuid: str
    summary: str
    file: str                # relative path from storage root
    delta_action: str        # create / update
    update_context: str      # reason + evidence


class BaseDetailPageHarness(BaseAgentHarness):
    """
    Second-level base for detail-page updater agents.

    Adds file-level orchestration: parse → call LLM → merge → diff → write.

    Subclasses implement:
      - parse_file(path) -> ParsedFile (BaseModel)
      - build_markdown(obj) -> str
      - apply_file_update(current_obj, proposal) -> new_obj
      - build_user_prompt(**context) -> str
      - format_proposal(proposal) -> str
    """

    def __init__(self, llm, system_prompt, output_schema, accessor, **kwargs):
        super().__init__(llm, system_prompt, output_schema, **kwargs)
        self.accessor = accessor
        # Detail review state
        self._targets: list[DetailTarget] = []
        self._target_idx: int = 0
        self._current_result: dict | None = None

    # ── Abstract (subclass must implement) ────────────────────

    @abstractmethod
    def parse_file(self, path: Path) -> BaseModel:
        """Parse a uuid.md detail file into a structured object."""
        ...

    @abstractmethod
    def build_markdown(self, obj: BaseModel) -> str:
        """Render a parsed object back to markdown."""
        ...

    @abstractmethod
    def apply_file_update(self, current_obj: BaseModel, proposal: BaseModel) -> BaseModel:
        """Merge a proposal into the current parsed object."""
        ...

    # ── apply_proposal is not used for detail harness ─────────
    def apply_proposal(self, proposal, **kwargs) -> list[str]:
        raise NotImplementedError("Use propose_page() / approve_page() instead.")

    # ── Concrete orchestration ────────────────────────────────

    def set_targets(self, targets: list[DetailTarget]):
        """Load a list of detail targets to process one by one."""
        self._targets = list(targets)
        self._target_idx = 0
        self._current_result = None
        self.reset()

    @property
    def current_target(self) -> DetailTarget | None:
        if self._target_idx < len(self._targets):
            return self._targets[self._target_idx]
        return None

    @property
    def remaining(self) -> int:
        return max(0, len(self._targets) - self._target_idx)

    def propose_page(self, diff_text: str, **extra_context) -> dict:
        """
        Call the LLM for the current target.
        Returns a result dict with target, objects, proposal, content, and patch.
        """
        target = self.current_target
        if target is None:
            raise ValueError("No more targets to process.")

        event_path = self.accessor.root / target.file
        current_raw = event_path.read_text(encoding="utf-8")
        current_obj = self.parse_file(event_path)

        # Call LLM via the base harness
        proposal = self.call(
            diff_text=diff_text,
            target=target,
            current_obj=current_obj,
            current_raw_markdown=current_raw,
            **extra_context,
        )

        new_obj = self.apply_file_update(current_obj, proposal)
        new_raw = self.build_markdown(new_obj)
        patch = build_unified_diff(current_raw, new_raw, target.file)

        self._current_result = {
            "target": target.model_dump(),
            "current_obj": current_obj.model_dump(),
            "proposal": proposal.model_dump(),
            "new_obj": new_obj.model_dump(),
            "old_content": current_raw,
            "new_content": new_raw,
            "patch": patch,
        }
        return self._current_result

    def review_page(self) -> str:
        """Format current page proposal for human review."""
        if self._current_result is None:
            return "⚠️  No page proposal to review. Run propose_page() first."
        return self.format_proposal(self._current_result["proposal"])

    def approve_page(self) -> str:
        """Write the updated markdown to disk and advance to next target."""
        if self._current_result is None:
            raise ValueError("No page proposal to approve.")

        target = self.current_target
        changed = self._current_result["proposal"]["changed"]

        msg = ""
        if changed:
            event_path = self.accessor.root / target.file
            event_path.write_text(self._current_result["new_content"], encoding="utf-8")
            msg = f"✅ Written: {target.file} — {self._current_result['proposal'].get('approval_message', '')}"
        else:
            msg = f"ℹ️  No changes for {target.uuid}. Skipping."

        self._advance()
        return msg

    def reject_page(self, feedback: str):
        """Record feedback for the current target, keep position."""
        if self._current_result is None:
            raise ValueError("No page proposal to reject.")
        self.reject(feedback)
        self._current_result = None

    def _advance(self):
        """Move to the next target and reset LLM state."""
        self._target_idx += 1
        self._current_result = None
        self.reset()

    def build_targets_from_accessor(self) -> list[DetailTarget]:
        """Build targets for all UUIDs currently in the accessor."""
        targets = []
        for uuid in self.accessor.list_uuids():
            rec = self.accessor.get(uuid)
            detail_path = self.accessor.spec.detail_path(self.accessor.root, uuid)
            targets.append(DetailTarget(
                uuid=uuid,
                summary=rec.get("summary", rec.get("display_name", uuid)),
                file=str(detail_path.relative_to(self.accessor.root)),
                delta_action="update",
                update_context="Full detail enrichment pass",
            ))
        return targets


print("✅ BaseDetailPageHarness + DetailTarget defined")

✅ BaseDetailPageHarness + DetailTarget defined


---

## Events Index Agent

Maintains the canonical `events.md` index. Proposes create/update/delete deltas.

In [32]:
# ── Events Index Agent: schema ────────────────────────────────────

class EventDeltaAction(str, Enum):
    CREATE = "create"
    UPDATE = "update"
    DELETE = "delete"


class EventDelta(BaseModel):
    """One proposed change to the events index."""

    action: EventDeltaAction = Field(
        description="Canon-level operation: create, update, or delete."
    )
    existing_event_uuid: Optional[str] = Field(
        default=None,
        description="UUID of the matched existing event for update/delete. Null for create."
    )
    when: str = Field(default="", description="Story-time placement.")
    chapters: str = Field(default="", description="Comma-separated chapter references.")
    summary: str = Field(default="", description="Short canonical description.")
    reason: str = Field(description="Grounded explanation of the decision.")
    evidence_from_diff: list[str] = Field(
        default_factory=list,
        description="Exact phrases from the diff."
    )

    @model_validator(mode="after")
    def validate_shape(self) -> "EventDelta":
        if self.action == EventDeltaAction.CREATE:
            if self.existing_event_uuid is not None:
                raise ValueError("existing_event_uuid must be null for create.")
            if not self.summary.strip():
                raise ValueError("summary is required for create.")
        elif self.action in (EventDeltaAction.UPDATE, EventDeltaAction.DELETE):
            if not self.existing_event_uuid:
                raise ValueError(f"existing_event_uuid is required for {self.action.value}.")
            if self.action == EventDeltaAction.UPDATE and not self.summary.strip():
                raise ValueError("summary is required for update.")
        return self


class EventAgentOutput(BaseModel):
    """Complete structured output from the Events Index Agent."""
    scan_summary: str = Field(description="Summary of diff changes at event-index level.")
    deltas: list[EventDelta] = Field(default_factory=list)


print("✅ Schema: EventDeltaAction, EventDelta, EventAgentOutput")

✅ Schema: EventDeltaAction, EventDelta, EventAgentOutput


In [33]:
# ── Events Index Agent: class ─────────────────────────────────────

EVENTS_INDEX_SYSTEM_PROMPT = """
You are the Events Index Agent.

Your single responsibility is to maintain the canonical events index for a story world.

You will receive:
1. The current contents of events.md — this includes the definitions of what an event is,
   what each field means, and the current index entries.
   READ THE DEFINITIONS SECTION IN events.md CAREFULLY. They are your operating contract.
2. An incoming manuscript diff showing what changed in the story text.

═══════════════════════════════════════════════════════════
CORE IDENTITY
═══════════════════════════════════════════════════════════

You are NOT a creative writer, story critic, prose summarizer, or theorist.
You are a careful canon librarian for events.

═══════════════════════════════════════════════════════════
METHOD
═══════════════════════════════════════════════════════════

Follow this exact two-pass process:

PASS 1 — SCAN: Read the diff top to bottom. For each chapter, list every
candidate event: a thing that happens, is discovered, is decided, or is revealed.

PASS 2 — CONSOLIDATE: Merge candidates that are the same event seen from
different angles. Remove candidates that are merely atmospheric detail,
scene-level motion, or emotional coloring. What remains are your final deltas.

Then match each final delta against events.md to decide create vs update vs delete.

═══════════════════════════════════════════════════════════
GRANULARITY
═══════════════════════════════════════════════════════════

Work at SCENE-LEVEL granularity, not beat-level or chapter-level.

One event = one bounded thing that happened, was discovered, or was decided.

Examples of CORRECT event granularity:
  ✓ "Mira receives a letter from her mother sealed with chapel wax"
  ✓ "A cloth bundle containing river stones, a cracked watch, and a toll house ledger page is found at the altar"
  ✓ "Sister Celine appears inside the chapel and confronts Mira about the silver key"

Examples of WRONG granularity (too fine):
  ✗ "Mira reaches the greenhouse" (scene-setting, not an event)
  ✗ "Arun hands Mira a cup of tea" (incidental action)

Examples of WRONG granularity (too coarse):
  ✗ "Mira and Arun visit the chapel and discover several things" (multiple events lumped)

A typical chapter-length diff should yield 3-8 events.

═══════════════════════════════════════════════════════════
DECISION DISCIPLINE
═══════════════════════════════════════════════════════════

- Prefer UPDATE over CREATE when the diff expands or clarifies an already indexed event.
- Do NOT create duplicates because the prose is richer or more specific.
- CREATE only when the diff introduces a distinct bounded happening not in the index.
- DELETE only when the diff clearly removes or invalidates a previously indexed event.

═══════════════════════════════════════════════════════════
UUID RULES
═══════════════════════════════════════════════════════════

- For CREATE: set existing_event_uuid to null. The system generates UUIDs.
- For UPDATE / DELETE: copy the exact UUID from events.md.
- NEVER invent or guess a UUID.

═══════════════════════════════════════════════════════════
GROUNDING RULES
═══════════════════════════════════════════════════════════

- Stay faithful to the diff and events.md.
- Do NOT invent hidden events or speculate.
- Every delta must have evidence_from_diff.

═══════════════════════════════════════════════════════════
FEEDBACK HANDLING
═══════════════════════════════════════════════════════════

If the user message includes a HISTORY section with previous attempts and
reviewer feedback:
1. Read the feedback carefully.
2. Incorporate it into your revised proposal.
3. Do NOT repeat the same mistakes.

═══════════════════════════════════════════════════════════
OUTPUT
═══════════════════════════════════════════════════════════

Return ONLY the structured output in the required schema.
If the diff implies no meaningful event-index change, return an empty deltas list.
"""

EVENTS_INDEX_USER_TEMPLATE = """Current events.md:

<events_md>
{events_md}
</events_md>

Incoming manuscript diff:

<diff>
{diff_text}
</diff>

Task:
1. Read the definitions in events.md — they are your operating contract.
2. Follow the two-pass method: SCAN all candidate events, then CONSOLIDATE.
3. For each final event, decide create / update / delete against the current index.
4. For updates and deletes, match to the existing UUID exactly.
5. Return the structured output.
"""


class EventIndexAgent(BaseAgentHarness):
    """Events Index Agent — proposes create/update/delete to events.md."""

    def __init__(self, llm, accessor: BaseLayerAccessor):
        super().__init__(
            llm=llm,
            system_prompt=EVENTS_INDEX_SYSTEM_PROMPT,
            output_schema=EventAgentOutput,
        )
        self.accessor = accessor

    def build_user_prompt(self, **ctx) -> str:
        return EVENTS_INDEX_USER_TEMPLATE.format(
            events_md=ctx["events_md"],
            diff_text=ctx["diff_text"],
        )

    def format_proposal(self, proposal: EventAgentOutput) -> str:
        lines = [
            "═" * 60,
            "EVENT INDEX AGENT PROPOSAL",
            "═" * 60,
            f"\nScan summary: {proposal.scan_summary}",
            f"Total deltas: {len(proposal.deltas)}",
        ]
        for i, delta in enumerate(proposal.deltas, 1):
            lines.append(f"\n{'─' * 50}")
            lines.append(f"Delta {i}: {delta.action.value.upper()}")
            lines.append(f"{'─' * 50}")
            if delta.existing_event_uuid:
                lines.append(f"  Target UUID:  {delta.existing_event_uuid}")
            if delta.action != EventDeltaAction.DELETE:
                lines.append(f"  When:         {delta.when}")
                lines.append(f"  Chapters:     {delta.chapters}")
                lines.append(f"  Summary:      {delta.summary}")
            lines.append(f"  Reason:       {delta.reason}")
            if delta.evidence_from_diff:
                lines.append(f"  Evidence:")
                for ev in delta.evidence_from_diff:
                    lines.append(f'    - "{ev}"')
        lines.append(f"\n{'═' * 60}")
        return "\n".join(lines)

    def apply_proposal(self, proposal: EventAgentOutput, **kwargs) -> list[str]:
        accessor = kwargs.get("accessor", self.accessor)
        actions = []
        for delta in proposal.deltas:
            if delta.action == EventDeltaAction.CREATE:
                new_uuid = accessor.apply_create({
                    "when": delta.when,
                    "chapters": delta.chapters,
                    "summary": delta.summary,
                })
                actions.append(f"CREATED: {new_uuid} — {delta.summary[:60]}")
            elif delta.action == EventDeltaAction.UPDATE:
                accessor.apply_update(delta.existing_event_uuid, {
                    "when": delta.when,
                    "chapters": delta.chapters,
                    "summary": delta.summary,
                })
                actions.append(f"UPDATED: {delta.existing_event_uuid} — {delta.summary[:60]}")
            elif delta.action == EventDeltaAction.DELETE:
                accessor.apply_delete(delta.existing_event_uuid)
                actions.append(f"DELETED: {delta.existing_event_uuid}")
        return actions


print("✅ EventIndexAgent defined")

✅ EventIndexAgent defined


### Events Index — Review Flow

1. **Call Agent** — runs the agent and stores the proposal
2. **Approve** — applies the proposal to `events.md` via the accessor
3. **Reject + Retry** — set your feedback, then re-run the Call Agent cell

In [34]:
# ── Instantiate Events Index Agent ────────────────────────────────

_diff_text = Path("example_story_change.diff").read_text(encoding="utf-8")
event_index_agent = EventIndexAgent(llm=llm, accessor=events)

print(f"✅ Loaded diff ({len(_diff_text)} chars)")
print(f"   Events accessor has {len(events.list_uuids())} existing event(s)")
print(f"   EventIndexAgent ready")

✅ Loaded diff (5907 chars)
   Events accessor has 7 existing event(s)
   EventIndexAgent ready


In [72]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: CALL THE EVENTS INDEX AGENT
# ══════════════════════════════════════════════════════════════════

FRESH_RUN = True

print(f"Calling Events Index Agent (attempt {event_index_agent.attempt + 1}, fresh={FRESH_RUN})...")

proposal = event_index_agent.call(
    fresh=FRESH_RUN,
    events_md=events.get_index_text(),
    diff_text=_diff_text,
)

print(f"✅ Agent returned {len(proposal.deltas)} delta(s).\n")
print(event_index_agent.review())

Calling Events Index Agent (attempt 2, fresh=True)...
✅ Agent returned 6 delta(s).

════════════════════════════════════════════════════════════
EVENT INDEX AGENT PROPOSAL
════════════════════════════════════════════════════════════

Scan summary: Five existing events were expanded with precise temporal details, direct dialogue, and additional object descriptions. One new event (Sister Celine's revelation) was added to reflect the manuscript's expanded narrative.
Total deltas: 6

──────────────────────────────────────────────────
Delta 1: UPDATE
──────────────────────────────────────────────────
  Target UUID:  evt_f72bc8fe0f29
  When:         Late June, 1998, before sunrise
  Chapters:     Chapter 7
  Summary:      Mira receives a letter from her mother sealed with chapel wax that mentions 'the one who went missing by the river'
  Reason:       Added precise arrival time (11:40 p.m.) and explicit quote from letter content
  Evidence:
    - "The letter from her mother had arrived at 11

In [36]:
# ══════════════════════════════════════════════════════════════════
# STEP 2a: APPROVE INDEX CHANGES
# ══════════════════════════════════════════════════════════════════

if not event_index_agent.has_proposal:
    print("⚠️  No proposal to approve. Run CALL AGENT first.")
else:
    actions = event_index_agent.approve()

    print("✅ APPROVED — changes applied:\n")
    for a in actions:
        print(f"  ✓ {a}")

    issues = events.validate()
    if issues:
        print(f"\n⚠️  Consistency issues:")
        for issue in issues:
            print(f"  - {issue}")
        for ra in events.repair():
            print(f"  ✓ {ra}")
    else:
        print(f"\n✅ Events layer consistent.")

    print(f"\nTotal events: {len(events.list_uuids())}")

✅ APPROVED — changes applied:

  ✓ UPDATED: evt_50b5adf351ae — Mira is carrying the silver key she found under the chapel f
  ✓ UPDATED: evt_f2e1969e1f2f — Mira and Arun discuss conflicting accounts of Elias's disapp
  ✓ UPDATED: evt_c43da1c48907 — A cloth bundle with river stones, a cracked watch stopped at
  ✓ UPDATED: evt_d23bb1179343 — A packet of letters bound in black thread dated July 3, 1988

✅ Events layer consistent.

Total events: 7


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 2b: REJECT WITH FEEDBACK
# ══════════════════════════════════════════════════════════════════

# ✏️  EDIT YOUR FEEDBACK HERE:
REJECTION_FEEDBACK = """
Your feedback here...
"""

if not event_index_agent.has_proposal:
    print("⚠️  No proposal to reject. Run CALL AGENT first.")
else:
    feedback = REJECTION_FEEDBACK.strip()
    if not feedback or feedback == "Your feedback here...":
        print("⚠️  Please edit REJECTION_FEEDBACK above before running.")
    else:
        event_index_agent.reject(feedback)
        print(f"✅ Feedback recorded for attempt {event_index_agent.attempt}.")
        print(f"\n→ Re-run CALL AGENT to get a revised proposal.")

---

## Elements Index Agent

Maintains the canonical `elements.md` index. Identifies durable story elements from diffs, matches against existing entries, and proposes new or updated elements.

In [82]:
# ── Elements Index Agent: schema ──────────────────────────────────

class ElementDecision(BaseModel):
    """One identified element from the diff."""

    display_name: str = Field(
        description="Canonical display name. Prefer the simplest stable name."
    )
    kind: Literal["person", "place", "item", "animal", "relationship", "concept", "group", "other"]

    aliases: list[str] = Field(
        default_factory=list,
        description="Alternative names explicitly supported by the diff or index."
    )
    identification_keys: list[str] = Field(
        default_factory=list,
        description="3-6 short recognition cues that distinguish this element."
    )
    snapshot: str = Field(
        description="2-3 sentence summary of what this element is in the story."
    )
    update_instruction: str = Field(
        description="Concrete instruction for the later uuid.md updater."
    )
    evidence_from_diff: list[str] = Field(
        default_factory=list,
        description="2-6 exact phrases from the diff justifying identification."
    )
    matched_existing_display_name: Optional[str] = Field(
        default=None,
        description="Exact display name from elements.md if matching an existing element."
    )
    matched_existing_uuid: Optional[str] = Field(
        default=None,
        description="UUID from elements.md if matching an existing element."
    )
    is_new: bool = Field(
        description="True only if no existing indexed element matches."
    )


class ElementsProposal(BaseModel):
    """Complete structured output from the Elements Index Agent."""

    diff_summary: str = Field(
        description="2-5 sentence summary of what the diff adds or changes."
    )
    rationale: str = Field(
        description="Short explanation of existing-vs-new decisions."
    )
    identified_elements: list[ElementDecision] = Field(
        default_factory=list,
        description="All relevant durable story elements in the diff."
    )
    approval_message: str = Field(
        description="Short human-facing summary of what will change if approved."
    )


print("✅ Schema: ElementDecision, ElementsProposal")

✅ Schema: ElementDecision, ElementsProposal


In [83]:
# ── Elements Index Agent: helpers ─────────────────────────────────

def normalize_lookup(text: str) -> str:
    """Normalize text for fuzzy matching."""
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def parse_elements_index(text: str) -> dict:
    """
    Parse elements.md into a lookup dict and records dict.
    Returns {"lookup": {normalized_str: record}, "records": {uuid: record}}
    """
    lookup = {}
    records = {}

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line.startswith("- "):
            continue
        parts = [p.strip() for p in line[2:].split("|")]
        if len(parts) < 5:
            continue

        kind, display_name, element_uuid, aliases_raw, keys_raw = parts[:5]
        aliases = [a.strip() for a in aliases_raw.split(",") if a.strip()]
        identification_keys = [k.strip() for k in keys_raw.split(";") if k.strip()]

        record = {
            "kind": kind,
            "display_name": display_name,
            "uuid": element_uuid,
            "aliases": aliases,
            "identification_keys": identification_keys,
        }
        records[element_uuid] = record

        for value in [display_name] + aliases + identification_keys:
            norm = normalize_lookup(value)
            if norm and norm not in lookup:
                lookup[norm] = record

    return {"lookup": lookup, "records": records}


def resolve_existing_element(existing_index: dict, candidate: dict) -> dict | None:
    """
    Try to match a candidate element to an existing indexed element.
    Matching order: display_name, aliases, identification_keys.
    """
    lookup = existing_index.get("lookup", {})
    candidate_values = (
        [candidate.get("display_name", "")]
        + candidate.get("aliases", [])
        + candidate.get("identification_keys", [])
    )
    for value in candidate_values:
        norm = normalize_lookup(value)
        if norm and norm in lookup:
            return lookup[norm]
    return None


def text_explicitly_mentions(text: str, phrase: str) -> bool:
    """Check whether a display name or alias appears as a bounded phrase in text."""
    phrase = phrase.strip()
    if not phrase:
        return False
    pattern = r"\b" + re.escape(phrase.lower()) + r"\b"
    return re.search(pattern, text.lower()) is not None


def find_explicit_index_mentions(existing_index: dict, diff_text: str) -> dict[str, list[dict]]:
    """Collect existing indexed elements that are explicitly named in the diff, grouped by kind."""
    mentions_by_kind = {}
    seen_uuids = set()
    for record in existing_index.get("records", {}).values():
        values = [record.get("display_name", "")] + record.get("aliases", [])
        if any(text_explicitly_mentions(diff_text, value) for value in values if value.strip()):
            uuid = record.get("uuid", "")
            if uuid and uuid not in seen_uuids:
                mentions_by_kind.setdefault(record.get("kind", "other"), []).append(record)
                seen_uuids.add(uuid)
    return mentions_by_kind


KINSHIP_CUE_PATTERN = re.compile(
    r"\b(mother|father|brother|sister|cousin|wife|husband|daughter|son|uncle|aunt|niece|nephew|grandmother|grandfather)\b",
    re.IGNORECASE,
)


def audit_elements_coverage(proposal: ElementsProposal, existing_index: dict, diff_text: str) -> list[str]:
    """Return targeted feedback when a proposal misses whole non-item categories present in the diff."""
    feedback = []
    mentions_by_kind = find_explicit_index_mentions(existing_index, diff_text)
    proposal_kinds = {elem.kind for elem in proposal.identified_elements}

    for kind in ("person", "place", "group", "relationship", "concept"):
        mentioned = mentions_by_kind.get(kind, [])
        if mentioned and kind not in proposal_kinds:
            sample = ", ".join(rec["display_name"] for rec in mentioned[:4])
            feedback.append(
                f"Coverage gap: the diff explicitly names existing {kind} element(s) {sample}, but your proposal returned no {kind} entries. Existing elements must still be returned when materially affected."
            )

    if KINSHIP_CUE_PATTERN.search(diff_text) and not any(
        elem.kind in {"person", "relationship"} for elem in proposal.identified_elements
    ):
        feedback.append(
            "Coverage gap: the diff contains kinship or relationship revelations. Include the affected people, and use kind='relationship' when the relationship itself should be indexed as a durable canonical element."
        )

    summary_text = f"{proposal.diff_summary} {proposal.rationale}"
    present_names = {normalize_lookup(elem.display_name) for elem in proposal.identified_elements}
    named_in_summary = []
    for record in existing_index.get("records", {}).values():
        name = record.get("display_name", "")
        if not name:
            continue
        if text_explicitly_mentions(summary_text, name) and normalize_lookup(name) not in present_names:
            named_in_summary.append(name)

    if named_in_summary:
        sample = ", ".join(named_in_summary[:4])
        feedback.append(
            f"Consistency gap: your diff_summary or rationale names {sample}, but those elements are missing from identified_elements. Every named durable entity in the summary or rationale should either appear in identified_elements or be explicitly ruled out as non-durable."
        )

    return feedback


print("✅ Element helpers: normalize_lookup, parse_elements_index, resolve_existing_element, audit_elements_coverage")

✅ Element helpers: normalize_lookup, parse_elements_index, resolve_existing_element, audit_elements_coverage


In [84]:
# ── Elements Index Agent: class ───────────────────────────────────

ELEMENTS_INDEX_SYSTEM_PROMPT = """
You are the Elements Index Agent.

Your job is to read:
1. the manuscript git diff
2. the current elements.md index

Then produce a proposal that identifies all durable story-relevant elements mentioned or materially affected by the diff.

A durable story-relevant element may be:
- a person, place, item, animal, relationship, group, concept, or other recurring entity

Do not index trivial background noise.
Do index elements that matter to understanding the story world, relationships, events, places, recurring objects, institutions, and groups.

Your output must be reader-friendly, auditable, and useful for downstream uuid.md updates.

═══════════════════════════════════════════════════════════
METHOD
═══════════════════════════════════════════════════════════

Follow this exact three-pass process:

PASS 1 — SCAN BY CATEGORY
Read the diff top to bottom and scan explicitly for:
- people
- places
- items
- groups / institutions
- concepts / rules
- relationships

List every durable candidate explicitly named or clearly referenced.

PASS 2 — CONSOLIDATE AND MATCH
Merge aliases and duplicate surface forms. Match candidates to existing elements.md entries when evidence is strong.

PASS 3 — COVERAGE CHECK
Before finalizing, verify that every materially affected named person, place, group, or relationship is represented in identified_elements.
Do not represent a person, group, or relationship change only through an item that reveals it.

═══════════════════════════════════════════════════════════
CORE IDENTITY
═══════════════════════════════════════════════════════════

You are NOT a creative writer or critic.
You are a world-model maintainer.

═══════════════════════════════════════════════════════════
RULES
═══════════════════════════════════════════════════════════

1. Do not judge the story.
Your job is to make the world model legible, not to correct the writer.

2. Prefer stable canonical naming.
Use the simplest stable canonical name, not a temporary descriptive phrase.
- "watch" over "cracked watch" (the crack is a property, not identity)
- "Elias" over "Elias Vale" if the surname is just a fuller mention

3. identification_keys must be real discriminators.
Bad: "Mira", "chapel"
Good: "carries the silver key", "missing figure from ten years earlier"

4. update_instruction must be concrete.
Bad: "Add new person entry for Mira"
Good: "Record that Mira receives her mother's letter, resumes carrying the silver key, recalls Elias, and leaves the chapel with the recovered letters."

5. snapshot must be useful.
2-3 sentences that explain the element's current role in the world model.

6. evidence_from_diff must be grounded.
Exact short phrases from the diff. Do not invent evidence.

7. Existing-vs-new matching.
Prefer matching an existing element when there is strong evidence from:
- exact name match, alias match, or clear conceptual match
Do not create a new element for a slightly different surface phrase.

8. Coverage.
Return all relevant durable elements, not just the first few.
Existing elements must still be returned when the diff changes their role, knowledge, relationship, interpretation, or story function.
Relationship revelations count as changes to both endpoints, and may also justify a relationship element when the relationship itself becomes canonically important.
Do not proxy a person, group, or relationship change only through an object such as a letter, ledger page, or key.

9. Do not create UUIDs.
If matching existing, copy the UUID. If new, set is_new=true.

10. Internal consistency.
If your diff_summary or rationale names a durable entity, that entity should appear in identified_elements unless you explicitly state why it is not durable enough to index.

═══════════════════════════════════════════════════════════
FEEDBACK HANDLING
═══════════════════════════════════════════════════════════

If the message includes HISTORY with previous attempts and feedback,
incorporate the feedback. Do not repeat the same mistakes.

═══════════════════════════════════════════════════════════
OUTPUT
═══════════════════════════════════════════════════════════

Return ONLY the structured output in the required schema.
"""

ELEMENTS_INDEX_USER_TEMPLATE = """Current elements.md:

<elements_md>
{elements_md}
</elements_md>

Incoming manuscript diff:

<diff>
{diff_text}
</diff>

Task:
1. Read elements.md as the operating contract for what counts as an element.
2. Scan the diff category by category: people, places, items, groups, concepts, relationships.
3. Identify all durable story elements in this diff, including existing elements whose role, knowledge, interpretation, or relationships are materially affected.
4. Match to existing elements.md entries when evidence is strong.
5. For new elements, set is_new=true and leave matched_existing_uuid empty.
6. Do not represent a person or relationship change only through the object that reveals it.
7. Return structured output.
"""


class ElementIndexAgent(BaseAgentHarness):
    """Elements Index Agent — identifies elements and proposes index updates."""

    def __init__(self, llm, accessor: BaseLayerAccessor):
        super().__init__(
            llm=llm,
            system_prompt=ELEMENTS_INDEX_SYSTEM_PROMPT,
            output_schema=ElementsProposal,
        )
        self.accessor = accessor

    def build_user_prompt(self, **ctx) -> str:
        return ELEMENTS_INDEX_USER_TEMPLATE.format(
            elements_md=ctx["elements_md"],
            diff_text=ctx["diff_text"],
        )

    def call(self, fresh: bool = False, **context):
        """Call once, audit category coverage, and retry once with targeted feedback if needed."""
        existing_index = parse_elements_index(context.get("elements_md", self.accessor.get_index_text()))
        proposal = None

        for pass_idx in range(2):
            proposal = super().call(fresh=fresh if pass_idx == 0 else False, **context)
            coverage_feedback = audit_elements_coverage(
                proposal,
                existing_index,
                context.get("diff_text", ""),
            )
            if not coverage_feedback or pass_idx == 1:
                return proposal
            self.reject("\n".join(coverage_feedback))

        return proposal

    def format_proposal(self, proposal: ElementsProposal) -> str:
        lines = [
            "═" * 60,
            "ELEMENTS INDEX AGENT PROPOSAL",
            "═" * 60,
            f"\nDiff summary: {proposal.diff_summary}",
            f"Rationale: {proposal.rationale}",
            f"Total elements: {len(proposal.identified_elements)}",
        ]
        for i, elem in enumerate(proposal.identified_elements, 1):
            status = "NEW" if elem.is_new else f"EXISTING → {elem.matched_existing_uuid}"
            lines.append(f"\n{'─' * 50}")
            lines.append(f"Element {i}: {elem.display_name} ({elem.kind}) [{status}]")
            lines.append(f"{'─' * 50}")
            if elem.aliases:
                lines.append(f"  Aliases:     {', '.join(elem.aliases)}")
            if elem.identification_keys:
                lines.append(f"  Keys:        {'; '.join(elem.identification_keys)}")
            lines.append(f"  Snapshot:    {elem.snapshot[:150]}...")
            lines.append(f"  Instruction: {elem.update_instruction[:150]}...")
            if elem.evidence_from_diff:
                lines.append(f"  Evidence:")
                for ev in elem.evidence_from_diff[:4]:
                    lines.append(f'    - "{ev}"')
        lines.append(f"\n{'═' * 60}")
        lines.append(f"Approval: {proposal.approval_message}")
        lines.append("═" * 60)
        return "\n".join(lines)

    def apply_proposal(self, proposal: ElementsProposal, **kwargs) -> list[str]:
        """
        Deterministic resolution: match LLM proposals to existing entries,
        generate UUIDs for new elements, apply via accessor.
        """
        accessor = kwargs.get("accessor", self.accessor)
        existing_index = parse_elements_index(accessor.get_index_text())
        actions = []

        for elem in proposal.identified_elements:
            elem_dict = elem.model_dump()

            # Try LLM-provided match first
            matched = None
            if elem_dict.get("matched_existing_uuid"):
                matched = existing_index["records"].get(elem_dict["matched_existing_uuid"])

            # Fallback to fuzzy resolution
            if matched is None:
                matched = resolve_existing_element(existing_index, elem_dict)

            if matched:
                # UPDATE existing element — refresh keys
                uuid = matched["uuid"]
                updates = {}
                if elem.identification_keys:
                    # Merge new keys with existing
                    existing_keys = matched.get("identification_keys", [])
                    merged = list(existing_keys)
                    existing_norm = {normalize_lookup(k) for k in existing_keys}
                    for k in elem.identification_keys:
                        if normalize_lookup(k) not in existing_norm:
                            merged.append(k)
                            existing_norm.add(normalize_lookup(k))
                    updates["identification_keys"] = "; ".join(merged)
                if elem.aliases:
                    existing_aliases = matched.get("aliases", [])
                    merged = list(existing_aliases)
                    existing_norm = {normalize_lookup(a) for a in existing_aliases}
                    for a in elem.aliases:
                        if normalize_lookup(a) not in existing_norm:
                            merged.append(a)
                            existing_norm.add(normalize_lookup(a))
                    updates["aliases"] = ", ".join(merged)
                if updates:
                    accessor.apply_update(uuid, updates)
                actions.append(f"MATCHED: {uuid} ({matched['display_name']}) — {elem.update_instruction[:60]}")
            else:
                # CREATE new element
                new_uuid = accessor.apply_create({
                    "kind": elem.kind,
                    "display_name": elem.display_name,
                    "aliases": ", ".join(elem.aliases) if elem.aliases else "",
                    "identification_keys": "; ".join(elem.identification_keys) if elem.identification_keys else "",
                })
                actions.append(f"CREATED: {new_uuid} ({elem.display_name}) — {elem.update_instruction[:60]}")

                # Register new entry in our local index so later elements can match it
                new_record = {
                    "kind": elem.kind,
                    "display_name": elem.display_name,
                    "uuid": new_uuid,
                    "aliases": elem.aliases,
                    "identification_keys": elem.identification_keys,
                }
                existing_index["records"][new_uuid] = new_record
                for value in [elem.display_name] + elem.aliases + elem.identification_keys:
                    norm = normalize_lookup(value)
                    if norm and norm not in existing_index["lookup"]:
                        existing_index["lookup"][norm] = new_record

        return actions


print("✅ ElementIndexAgent defined")

✅ ElementIndexAgent defined


### Elements Index — Review Flow

1. **Call Agent** — runs the elements agent and stores the proposal
2. **Approve** — matches elements, creates new entries via accessor
3. **Reject + Retry** — set your feedback, then re-run

In [93]:
# ── Instantiate Elements Index Agent ──────────────────────────────

element_index_agent = ElementIndexAgent(llm=llm, accessor=elements)

print(f"✅ ElementIndexAgent ready")
print(f"   Elements accessor has {len(elements.list_uuids())} existing element(s)")

✅ ElementIndexAgent ready
   Elements accessor has 0 existing element(s)


In [94]:
# ══════════════════════════════════════════════════════════════════
# STEP 1: CALL THE ELEMENTS INDEX AGENT
# ══════════════════════════════════════════════════════════════════

FRESH_RUN = True

print(f"Calling Elements Index Agent (attempt {element_index_agent.attempt + 1}, fresh={FRESH_RUN})...")

proposal = element_index_agent.call(
    fresh=FRESH_RUN,
    elements_md=elements.get_index_text(),
    diff_text=_diff_text,
)

print(f"✅ Agent returned {len(proposal.identified_elements)} element(s).\n")
print(element_index_agent.review())

Calling Elements Index Agent (attempt 1, fresh=True)...
✅ Agent returned 12 element(s).

════════════════════════════════════════════════════════════
ELEMENTS INDEX AGENT PROPOSAL
════════════════════════════════════════════════════════════

Diff summary: The diff adds detailed temporal markers (dates/times), expands the chapel's physical description, introduces Sister Celine as a hidden figure, reveals the silver key's significance, and clarifies Elias's relationship to Mira through the toll house ledger. New objects like the cracked watch and cloth bundle with evidence are added.
Rationale: All elements are new as elements.md is currently empty. Created entries for all durable entities including people, places, items, and relationships directly mentioned or materially affected by the diff. Avoided proxying person/relationship changes through objects like letters or keys.
Total elements: 12

──────────────────────────────────────────────────
Element 1: Mira (person) [NEW]
────────────

In [95]:
# ══════════════════════════════════════════════════════════════════
# STEP 2a: APPROVE ELEMENTS INDEX
# ══════════════════════════════════════════════════════════════════

if not element_index_agent.has_proposal:
    print("⚠️  No proposal to approve. Run CALL AGENT first.")
else:
    actions = element_index_agent.approve()

    print("✅ APPROVED — changes applied:\n")
    for a in actions:
        print(f"  ✓ {a}")

    issues = elements.validate()
    if issues:
        print(f"\n⚠️  Consistency issues:")
        for issue in issues:
            print(f"  - {issue}")
        for ra in elements.repair():
            print(f"  ✓ {ra}")
    else:
        print(f"\n✅ Elements layer consistent.")

    print(f"\nTotal elements: {len(elements.list_uuids())}")

✅ APPROVED — changes applied:

  ✓ CREATED: elt_45d617e4531b (Mira) — Record that Mira receives her mother's letter, resumes carry
  ✓ CREATED: elt_b973f2e09131 (Arun) — Record that Arun shares knowledge about Elias's disappearanc
  ✓ CREATED: elt_9f0d7e04852f (Elias) — Record that Elias is Mira's mother's brother (not cousin) as
  ✓ CREATED: elt_03e8d4548117 (Saint Alder Chapel) — Record that the chapel contains the silver key, a cloth bund
  ✓ CREATED: elt_bd49df28a5e7 (Silver Key) — Record that the silver key was found under chapel floorboard
  ✓ CREATED: elt_4f2c3df00e4e (Toll House Ledger Page) — Record that the ledger page reveals Elias is Mira's mother's
  ✓ CREATED: elt_858965cdeeb9 (Sister Celine) — Record that Sister Celine buried Nani, vanished from records
  ✓ CREATED: elt_8f8a9d9a7e50 (Cloth Bundle) — Record that the cloth bundle contains river stones, a cracke
  ✓ CREATED: elt_e2bc7b804e58 (Cracked Watch) — Record that the watch has a cracked face stopped at 2:12 a.m
  ✓ 

In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 2b: REJECT WITH FEEDBACK
# ══════════════════════════════════════════════════════════════════

# ✏️  EDIT YOUR FEEDBACK HERE:
ELEMENT_REJECTION_FEEDBACK = """
Your feedback here...
"""

if not element_index_agent.has_proposal:
    print("⚠️  No proposal to reject. Run CALL AGENT first.")
else:
    feedback = ELEMENT_REJECTION_FEEDBACK.strip()
    if not feedback or feedback == "Your feedback here...":
        print("⚠️  Please edit ELEMENT_REJECTION_FEEDBACK above before running.")
    else:
        element_index_agent.reject(feedback)
        print(f"✅ Feedback recorded for attempt {element_index_agent.attempt}.")
        print(f"\n→ Re-run CALL AGENT to get a revised proposal.")

---

## Element Detail Agent

After the index is approved, enriches each `elt_<uuid>.md` detail file one at a time.


In [96]:
# ── Element Detail Agent: schema ──────────────────────────────────

class ChronologyBlock(BaseModel):
    """One chronology heading with element-centered bullet entries."""
    heading: str
    entries: list[str] = Field(default_factory=list)


class ParsedElementFile(BaseModel):
    """In-memory representation of one elt_<uuid>.md detail file."""
    uuid: str = ""
    kind: str = ""
    canonical_name: str = ""
    aliases: list[str] = Field(default_factory=list)
    identification_keys: list[str] = Field(default_factory=list)
    core_understanding: str = ""
    stable_profile_lines: list[str] = Field(default_factory=list)
    interpretation_lines: list[str] = Field(default_factory=list)
    knowledge_lines: list[str] = Field(default_factory=list)
    chronology_blocks: list[ChronologyBlock] = Field(default_factory=list)
    open_threads_lines: list[str] = Field(default_factory=list)


class ChronologyBlockUpdate(BaseModel):
    heading: str = Field(
        description="High-level chronology bucket like 'Before current narrative' or 'Chapter 8 — June 28, 1998'."
    )
    entries: list[str] = Field(
        default_factory=list,
        description="1-4 concise bullets from this element's perspective."
    )


class ElementFileUpdateProposal(BaseModel):
    """Structured proposal for updating one elt_<uuid>.md detail file."""
    changed: bool = Field(description="Whether this file should change.")
    rationale: str = Field(description="Why these changes are proposed.")
    core_understanding_replacement: Optional[str] = Field(
        default=None,
        description="Replacement for Core Understanding, or null."
    )
    stable_profile_to_add: list[str] = Field(default_factory=list)
    stable_profile_to_remove: list[str] = Field(default_factory=list)
    interpretation_to_add: list[str] = Field(default_factory=list)
    interpretation_to_remove: list[str] = Field(default_factory=list)
    knowledge_to_add: list[str] = Field(default_factory=list)
    knowledge_to_remove: list[str] = Field(default_factory=list)
    chronology_blocks_to_add: list[ChronologyBlockUpdate] = Field(default_factory=list)
    open_threads_to_add: list[str] = Field(default_factory=list)
    open_threads_to_remove: list[str] = Field(default_factory=list)
    approval_message: str = Field(description="Short human-facing summary.")


print("✅ Schema: ChronologyBlock, ParsedElementFile, ChronologyBlockUpdate, ElementFileUpdateProposal")


✅ Schema: ChronologyBlock, ParsedElementFile, ChronologyBlockUpdate, ElementFileUpdateProposal


In [97]:
# ── Element Detail Agent: helpers ─────────────────────────────────

def split_csv_field(value: str) -> list[str]:
    if not value or value == "-":
        return []
    return [item.strip() for item in value.split(",") if item.strip()]


def split_semicolon_field(value: str) -> list[str]:
    if not value or value == "-":
        return []
    return [item.strip() for item in value.split(";") if item.strip()]


def clean_element_line(value: str) -> str:
    cleaned = re.sub(r"^\s*-\s*", "", (value or "").strip())
    return "" if not cleaned or cleaned == "TBD" else cleaned


def clean_element_paragraph(value: str) -> str:
    text = (value or "").strip()
    if not text or text in {"TBD", "- TBD"}:
        return ""
    if "\n" not in text:
        return clean_element_line(text)
    return text


def clean_element_lines(values: list[str]) -> list[str]:
    cleaned = []
    seen = set()
    for value in values:
        item = clean_element_line(value)
        norm = normalize_line(item) if item else ""
        if item and norm not in seen:
            cleaned.append(item)
            seen.add(norm)
    return cleaned


def parse_chronology_blocks(section_text: str) -> list[ChronologyBlock]:
    section_text = section_text.strip()
    if not section_text or section_text in {"TBD", "- TBD"}:
        return []

    blocks: list[ChronologyBlock] = []
    current_heading: str | None = None
    current_entries: list[str] = []

    for raw in section_text.splitlines():
        line = raw.strip()
        if not line:
            continue
        if line.startswith("### "):
            if current_heading and current_entries:
                blocks.append(ChronologyBlock(heading=current_heading, entries=clean_element_lines(current_entries)))
            current_heading = line[4:].strip()
            current_entries = []
            continue
        if line.startswith("- "):
            entry = clean_element_line(line)
            if entry:
                if current_heading is None:
                    current_heading = "Ungrouped"
                current_entries.append(entry)

    if current_heading and current_entries:
        blocks.append(ChronologyBlock(heading=current_heading, entries=clean_element_lines(current_entries)))

    return [block for block in blocks if block.heading.strip() and block.entries]


def merge_element_section_lines(
    current_lines: list[str],
    to_add: list[str],
    to_remove: list[str],
) -> list[str]:
    result = clean_element_lines(current_lines)
    remove_set = {normalize_line(item) for item in clean_element_lines(to_remove)}
    result = [item for item in result if normalize_line(item) not in remove_set]
    existing_set = {normalize_line(item) for item in result}
    for item in clean_element_lines(to_add):
        norm = normalize_line(item)
        if norm not in existing_set:
            result.append(item)
            existing_set.add(norm)
    return result


def merge_chronology_blocks(
    current_blocks: list[ChronologyBlock],
    blocks_to_add: list[ChronologyBlockUpdate],
) -> list[ChronologyBlock]:
    merged: dict[str, ChronologyBlock] = {}

    for block in current_blocks:
        heading = block.heading.strip()
        entries = clean_element_lines(block.entries)
        if heading and entries:
            merged[heading] = ChronologyBlock(heading=heading, entries=entries)

    for block in blocks_to_add:
        heading = block.heading.strip()
        entries = clean_element_lines(block.entries)
        if not heading or not entries:
            continue
        if heading not in merged:
            merged[heading] = ChronologyBlock(heading=heading, entries=[])
        existing = {normalize_line(item) for item in merged[heading].entries}
        for item in entries:
            norm = normalize_line(item)
            if norm not in existing:
                merged[heading].entries.append(item)
                existing.add(norm)

    return list(merged.values())


print("✅ Element detail helpers: split fields, cleaning, chronology parsing, merge helpers")


✅ Element detail helpers: split fields, cleaning, chronology parsing, merge helpers


In [98]:
# ── Element Detail Agent: class ───────────────────────────────────

ELEMENT_DETAIL_SYSTEM_PROMPT = """
You are the Element Page Updater.

Your job is to update exactly one element detail page (elt_<uuid>.md).

You receive:
1. the manuscript git diff
2. the current elements.md index
3. the current events.md index (for chronology grounding)
4. the current raw markdown for this elt_<uuid>.md file
5. the current parsed element object
6. the current target metadata

Your task is to propose updates for this one element page only.

This page is not a scene transcript or a flat extraction log.
It is an element-centered dossier that explains who or what this element is,
what stable facts matter, what the recent diff changes, what the element knows,
and how its chronology should be organized.

Rules:
- You are NOT a creative writer. You are a careful canon librarian.
- Stay local to this one element. Do not propose changes to other elements.
- The diff is the source of truth for what changed.
- The page may already have content beyond TBD. Preserve useful existing content unless the diff clearly adds, clarifies, revises, or invalidates it.
- Do not invent unsupported facts, motives, or relationships.
- Stable Profile is for durable truths: roles, relationships, possessions, repeated associations, and stable traits.
- Do not fill Stable Profile with one-off scene actions that belong in chronology.
- Interpretation should explain what the recent diff suggests this means for the element.
- Knowledge / Beliefs / Uncertainties should stay grounded in this element's own perspective when applicable.
- Element-Centered Chronology should be grouped by era, chapter, or date. Prefer headings such as 'Before current narrative' or 'Chapter 8 — June 28, 1998'.
- Do not flood chronology with minute-by-minute bullets unless exact timing is causally important.
- Use to_remove only when existing content is clearly wrong or superseded by the diff.

Section guidance:

1. Core Understanding
Write 2-4 sentences explaining what this element is in the story world, what role it plays, and what deeper significance the recent diff sharpens.
This should not read like a scene recap.

2. Stable Profile
Bullet points for durable facts only:
- roles
- relationships
- possessions
- recurring associations
- stable traits

3. Interpretation
Bullet points about what the recent diff suggests this means for the element:
- emotional pressure
- secrecy
- motive
- role in unfolding mystery
- relationship implications

4. Knowledge / Beliefs / Uncertainties
Track what this element appears to know, suspect, misunderstand, or still not know.

5. Element-Centered Chronology
Group chronology into a small number of headings.
Within each heading, write 1-4 concise bullets from this element's perspective.

6. Open Threads
Bullet points for unresolved questions, pressures, or unknowns that remain active for this element.

Output rules:
- Return only the structured proposal.
- Be concrete, concise, interpretive, and auditable.
- Do not emit raw markdown patches or full file content.
- If the page already has good content and the diff changes nothing relevant, set changed=false.
"""

ELEMENT_DETAIL_USER_TEMPLATE = """Current elements.md index:
<elements_md>
{elements_md}
</elements_md>

Current events.md index:
<events_md>
{events_md}
</events_md>

Element update target:
- UUID: {uuid}
- Display name: {display_name}
- Kind: {kind}
- Delta action: {delta_action}
- Context: {update_context}

Current parsed element object:
<current_object>
{current_object_json}
</current_object>

Current raw elt_<uuid>.md markdown:
<current_markdown>
{current_raw_markdown}
</current_markdown>

Source manuscript diff:
<diff>
{diff_text}
</diff>

Task:
Propose updates for this one element detail page only.
Deepen the dossier while preserving accurate existing content.
Keep chronology grouped by era, chapter, or date unless exact timing is causally important.
"""


class ElementDetailAgent(BaseDetailPageHarness):
    """Element Detail Agent — enriches elt_<uuid>.md files one at a time."""

    def __init__(self, llm, accessor: BaseLayerAccessor, events_accessor: BaseLayerAccessor | None = None):
        super().__init__(
            llm=llm,
            system_prompt=ELEMENT_DETAIL_SYSTEM_PROMPT,
            output_schema=ElementFileUpdateProposal,
            accessor=accessor,
        )
        self.events_accessor = events_accessor

    def parse_file(self, path: Path) -> ParsedElementFile:
        text = Path(path).read_text(encoding="utf-8")
        ident = extract_section(text, "## Identification", "## Core Understanding")

        title_match = re.search(r"^#\s+(.+)$", text, flags=re.MULTILINE)
        canonical_name = title_match.group(1).strip() if title_match else ""
        uuid = kind = ""
        aliases: list[str] = []
        identification_keys: list[str] = []

        for raw in ident.splitlines():
            line = raw.strip()
            if line.startswith("- UUID:"):
                uuid = parse_bullet_field(line, "- UUID:")
            elif line.startswith("- Type:"):
                kind = parse_bullet_field(line, "- Type:")
            elif line.startswith("- Canonical name:"):
                canonical_name = parse_bullet_field(line, "- Canonical name:") or canonical_name
            elif line.startswith("- Aliases:"):
                aliases = split_csv_field(parse_bullet_field(line, "- Aliases:"))
            elif line.startswith("- Identification keys:"):
                identification_keys = split_semicolon_field(parse_bullet_field(line, "- Identification keys:"))

        core_understanding = extract_section(text, "## Core Understanding", "## Stable Profile")
        stable_profile = extract_section(text, "## Stable Profile", "## Interpretation")
        interpretation = extract_section(text, "## Interpretation", "## Knowledge / Beliefs / Uncertainties")
        knowledge = extract_section(text, "## Knowledge / Beliefs / Uncertainties", "## Element-Centered Chronology")
        chronology = extract_section(text, "## Element-Centered Chronology", "## Open Threads")
        open_threads = extract_section(text, "## Open Threads", None)

        # Backward compatibility with earlier element-page experiments.
        if not core_understanding:
            core_understanding = extract_section(text, "## Snapshot", "## Attributes")
        if not stable_profile:
            stable_profile = extract_section(text, "## Attributes", "## Timeline")

        chronology_blocks = parse_chronology_blocks(chronology)
        if not chronology_blocks:
            old_timeline = extract_section(text, "## Timeline", None)
            old_timeline_lines = clean_element_lines(parse_bullet_lines(old_timeline))
            if old_timeline_lines:
                chronology_blocks = [ChronologyBlock(heading="Imported Timeline", entries=old_timeline_lines)]

        return ParsedElementFile(
            uuid=uuid,
            kind=kind,
            canonical_name=canonical_name,
            aliases=aliases,
            identification_keys=identification_keys,
            core_understanding=clean_element_paragraph(core_understanding),
            stable_profile_lines=clean_element_lines(parse_bullet_lines(stable_profile)),
            interpretation_lines=clean_element_lines(parse_bullet_lines(interpretation)),
            knowledge_lines=clean_element_lines(parse_bullet_lines(knowledge)),
            chronology_blocks=chronology_blocks,
            open_threads_lines=clean_element_lines(parse_bullet_lines(open_threads)),
        )

    def build_markdown(self, obj: ParsedElementFile) -> str:
        aliases_str = ", ".join(clean_element_lines(obj.aliases)) if obj.aliases else "-"
        keys_str = "; ".join(clean_element_lines(obj.identification_keys)) if obj.identification_keys else "-"

        lines = [
            f"# {obj.canonical_name}",
            "",
            "## Identification",
            f"- UUID: {obj.uuid}",
            f"- Type: {obj.kind}",
            f"- Canonical name: {obj.canonical_name}",
            f"- Aliases: {aliases_str}",
            f"- Identification keys: {keys_str}",
            "",
            "## Core Understanding",
            obj.core_understanding.strip() if obj.core_understanding.strip() else "- TBD",
            "",
            "## Stable Profile",
        ]
        lines.extend(bullet_section(clean_element_lines(obj.stable_profile_lines)))
        lines.extend(["", "## Interpretation"])
        lines.extend(bullet_section(clean_element_lines(obj.interpretation_lines)))
        lines.extend(["", "## Knowledge / Beliefs / Uncertainties"])
        lines.extend(bullet_section(clean_element_lines(obj.knowledge_lines)))
        lines.extend(["", "## Element-Centered Chronology"])

        chronology_blocks = []
        for block in obj.chronology_blocks:
            entries = clean_element_lines(block.entries)
            heading = block.heading.strip()
            if heading and entries:
                chronology_blocks.append(ChronologyBlock(heading=heading, entries=entries))

        if chronology_blocks:
            for block in chronology_blocks:
                lines.append(f"### {block.heading}")
                for entry in block.entries:
                    lines.append(f"- {entry}")
                lines.append("")
            if lines[-1] == "":
                lines.pop()
        else:
            lines.append("- TBD")

        lines.extend(["", "## Open Threads"])
        lines.extend(bullet_section(clean_element_lines(obj.open_threads_lines)))
        lines.append("")
        return "\n".join(lines)

    def apply_file_update(self, current_obj: ParsedElementFile, proposal: ElementFileUpdateProposal) -> ParsedElementFile:
        if not proposal.changed:
            return current_obj.model_copy(deep=True)

        new = current_obj.model_copy(deep=True)
        replacement = clean_element_paragraph(proposal.core_understanding_replacement or "")
        if replacement:
            new.core_understanding = replacement
        new.stable_profile_lines = merge_element_section_lines(
            new.stable_profile_lines,
            proposal.stable_profile_to_add,
            proposal.stable_profile_to_remove,
        )
        new.interpretation_lines = merge_element_section_lines(
            new.interpretation_lines,
            proposal.interpretation_to_add,
            proposal.interpretation_to_remove,
        )
        new.knowledge_lines = merge_element_section_lines(
            new.knowledge_lines,
            proposal.knowledge_to_add,
            proposal.knowledge_to_remove,
        )
        new.chronology_blocks = merge_chronology_blocks(
            new.chronology_blocks,
            proposal.chronology_blocks_to_add,
        )
        new.open_threads_lines = merge_element_section_lines(
            new.open_threads_lines,
            proposal.open_threads_to_add,
            proposal.open_threads_to_remove,
        )
        return new

    def build_user_prompt(self, **ctx) -> str:
        target = ctx["target"]
        current_obj = ctx["current_obj"]
        events_md = ctx.get("events_md")
        if events_md is None and self.events_accessor is not None:
            events_md = self.events_accessor.get_index_text()
        events_md = events_md or "[events index unavailable]"
        return ELEMENT_DETAIL_USER_TEMPLATE.format(
            elements_md=self.accessor.get_index_text(),
            events_md=events_md,
            uuid=target.uuid,
            display_name=current_obj.canonical_name or target.summary,
            kind=current_obj.kind,
            delta_action=target.delta_action,
            update_context=target.update_context,
            current_object_json=current_obj.model_dump_json(indent=2),
            current_raw_markdown=ctx["current_raw_markdown"],
            diff_text=ctx["diff_text"],
        )

    def format_proposal(self, proposal) -> str:
        p = proposal if isinstance(proposal, dict) else proposal.model_dump() if hasattr(proposal, "model_dump") else proposal
        target = self.current_target
        lines = [
            f"━━━ Element Detail Update: {target.uuid if target else '?'} ({target.summary if target else '?'}) ━━━",
            f"Changed: {p.get('changed', '?')}",
            f"Rationale: {p.get('rationale', '')}",
            "",
        ]
        if p.get("core_understanding_replacement"):
            lines.append("Core Understanding (replacement):")
            lines.append(f"  {p['core_understanding_replacement'][:400]}")
            lines.append("")

        for name, add_key, rm_key in [
            ("Stable Profile", "stable_profile_to_add", "stable_profile_to_remove"),
            ("Interpretation", "interpretation_to_add", "interpretation_to_remove"),
            ("Knowledge / Beliefs / Uncertainties", "knowledge_to_add", "knowledge_to_remove"),
            ("Open Threads", "open_threads_to_add", "open_threads_to_remove"),
        ]:
            adds = p.get(add_key, [])
            rms = p.get(rm_key, [])
            if adds or rms:
                lines.append(f"{name}:")
                for item in adds:
                    lines.append(f"  + {item}")
                for item in rms:
                    lines.append(f"  - {item}")
                lines.append("")

        chronology_adds = p.get("chronology_blocks_to_add", [])
        if chronology_adds:
            lines.append("Element-Centered Chronology:")
            for block in chronology_adds:
                heading = block.get("heading", "") if isinstance(block, dict) else getattr(block, "heading", "")
                entries = block.get("entries", []) if isinstance(block, dict) else getattr(block, "entries", [])
                lines.append(f"  + {heading}")
                for entry in entries:
                    lines.append(f"    - {entry}")
            lines.append("")

        lines.append(f"Approval: {p.get('approval_message', '')}")
        return "\n".join(lines)


print("✅ ElementDetailAgent defined")


✅ ElementDetailAgent defined


### Element Detail — Review Flow

1. **Build targets** — queue elements for detail enrichment
2. **Call updater** — processes one element at a time
3. **Approve** — writes updated markdown to disk, advances
4. **Reject + Retry** — provide feedback, re-run updater


In [99]:
# ── Instantiate Element Detail Agent ──────────────────────────────

element_detail_agent = ElementDetailAgent(
    llm=llm,
    accessor=elements,
    events_accessor=events,
)

print("✅ ElementDetailAgent ready")


✅ ElementDetailAgent ready


In [100]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: BUILD ELEMENT DETAIL TARGETS
# Run AFTER approving the elements index (Step 2a).
# ══════════════════════════════════════════════════════════════════

targets = element_detail_agent.build_targets_from_accessor()
element_detail_agent.set_targets(targets)

print(f"✅ {len(targets)} element(s) queued for detail updates:")
for i, target in enumerate(targets):
    print(f"  [{i}] {target.uuid} — {target.summary[:60]}")


✅ 9 element(s) queued for detail updates:
  [0] elt_45d617e4531b — Mira
  [1] elt_b973f2e09131 — Arun
  [2] elt_9f0d7e04852f — Elias
  [3] elt_03e8d4548117 — Saint Alder Chapel
  [4] elt_bd49df28a5e7 — Silver Key
  [5] elt_4f2c3df00e4e — Toll House Ledger Page
  [6] elt_858965cdeeb9 — Sister Celine
  [7] elt_8f8a9d9a7e50 — Cloth Bundle
  [8] elt_e2bc7b804e58 — Cracked Watch


In [108]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: CALL ELEMENT DETAIL UPDATER (one at a time)
# ══════════════════════════════════════════════════════════════════

target = element_detail_agent.current_target
if target is None:
    print("✅ All element detail pages have been processed.")
else:
    print(f"Calling Element Detail Agent for {target.uuid}")
    print(f"  Name: {target.summary[:80]}")
    print(f"  Attempt: {element_detail_agent.attempt + 1}")
    print()

    result = element_detail_agent.propose_page(diff_text=_diff_text)
    print(element_detail_agent.review_page())

    if result.get("patch"):
        print("\nPatch preview:")
        print(result["patch"][:2000])


Calling Element Detail Agent for elt_b973f2e09131
  Name: Arun
  Attempt: 3

━━━ Element Detail Update: elt_b973f2e09131 (Arun) ━━━
Changed: True
Rationale: The diff reveals Arun's layered role as both practical ally and keeper of manipulated family histories. His greenhouse work symbolizes controlled growth, mirroring his careful management of truth about Elias. The revised proposal adds depth to his internal conflict between duty to Mira and unresolved personal stakes in the key's purpose.

Core Understanding (replacement):
  Arun operates as a pragmatic collaborator and gatekeeper of distorted family histories. His greenhouse work and key-related knowledge position him as both Mira's stabilizing force and a participant in the same systemic memory manipulation he seeks to uncover.

Stable Profile:
  + Greenhouse manager with symbolic role in controlled growth
  + Custodian of three conflicting Elias narratives
  + Keyholder of 1988 chapel discovery (age 13)

Interpretation:
  + Manag

In [109]:
# ══════════════════════════════════════════════════════════════════
# STEP 5a: APPROVE ELEMENT DETAIL UPDATE
# ══════════════════════════════════════════════════════════════════

if element_detail_agent._current_result is None:
    print("⚠️  No proposal to approve. Run CALL DETAIL UPDATER first.")
else:
    msg = element_detail_agent.approve_page()
    print(msg)

    issues = elements.validate()
    if issues:
        print("\n⚠️  Consistency issues:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("\n✅ Elements layer consistent.")

    remaining = element_detail_agent.remaining
    if remaining > 0:
        print(f"\n→ {remaining} element(s) remaining. Re-run CALL DETAIL UPDATER.")
    else:
        print("\n✅ All element detail pages processed.")


✅ Written: story/elements/elt_b973f2e09131.md — Enhanced Arun's dossier with layered analysis of greenhouse symbolism, memory manipulation dynamics, and expanded chronology details from Chapter 7-8

✅ Elements layer consistent.

→ 7 element(s) remaining. Re-run CALL DETAIL UPDATER.


In [106]:
# ══════════════════════════════════════════════════════════════════
# STEP 5b: REJECT ELEMENT DETAIL UPDATE
# ══════════════════════════════════════════════════════════════════

# ✏️  EDIT YOUR FEEDBACK HERE:
ELEMENT_DETAIL_REJECTION_FEEDBACK = """
Arun is a deeper character lets make the details more intricate
"""

if element_detail_agent._current_result is None:
    print("⚠️  No proposal to reject. Run CALL DETAIL UPDATER first.")
else:
    feedback = ELEMENT_DETAIL_REJECTION_FEEDBACK.strip()
    if not feedback or feedback == "Your feedback here...":
        print("⚠️  Please edit ELEMENT_DETAIL_REJECTION_FEEDBACK above before running.")
    else:
        element_detail_agent.reject_page(feedback)
        print(f"✅ Feedback recorded for attempt {element_detail_agent.attempt}.")
        print("\n→ Re-run CALL DETAIL UPDATER to get a revised proposal.")


✅ Feedback recorded for attempt 1.

→ Re-run CALL DETAIL UPDATER to get a revised proposal.


---

## Event Detail Agent

After the index is approved, enriches each `evt_<uuid>.md` detail file one at a time.

In [37]:
# ── Event Detail Agent: schema ────────────────────────────────────

class ParsedEventFile(BaseModel):
    """In-memory representation of one evt_<uuid>.md detail file."""
    uuid: str = ""
    when: str = ""
    chapters: str = ""
    summary: str = ""
    core_understanding: str = ""
    causal_context_lines: list[str] = Field(default_factory=list)
    consequences_lines: list[str] = Field(default_factory=list)
    participants_lines: list[str] = Field(default_factory=list)
    evidence_lines: list[str] = Field(default_factory=list)
    open_threads_lines: list[str] = Field(default_factory=list)


class EventFileUpdateProposal(BaseModel):
    """Structured proposal for updating one evt_<uuid>.md detail file."""
    changed: bool = Field(description="Whether this file should change.")
    rationale: str = Field(description="Why these changes are proposed.")
    core_understanding_replacement: Optional[str] = Field(
        default=None, description="Replacement for Core Understanding, or null."
    )
    causal_context_to_add: list[str] = Field(default_factory=list)
    causal_context_to_remove: list[str] = Field(default_factory=list)
    consequences_to_add: list[str] = Field(default_factory=list)
    consequences_to_remove: list[str] = Field(default_factory=list)
    participants_to_add: list[str] = Field(default_factory=list)
    participants_to_remove: list[str] = Field(default_factory=list)
    evidence_to_add: list[str] = Field(default_factory=list)
    evidence_to_remove: list[str] = Field(default_factory=list)
    open_threads_to_add: list[str] = Field(default_factory=list)
    open_threads_to_remove: list[str] = Field(default_factory=list)
    approval_message: str = Field(description="Short human-facing summary.")


print("✅ Schema: ParsedEventFile, EventFileUpdateProposal")

✅ Schema: ParsedEventFile, EventFileUpdateProposal


In [38]:
# ── Event Detail Agent: class ─────────────────────────────────────

EVENT_DETAIL_SYSTEM_PROMPT = """
You are the Event Page Updater.

Your job is to update exactly one event detail page (evt_<uuid>.md).

You receive:
1. the manuscript git diff
2. the current events.md index (for cross-reference)
3. the current raw markdown for this evt_<uuid>.md file
4. the current parsed event object

Your task is to propose updates for this one event page only.

This page is not a scene transcript or a blow-by-blow log.
It is an event-centered dossier that explains what happened, why it matters,
and what remains unresolved.

Rules:
- You are NOT a creative writer. You are a careful canon librarian.
- Stay local to this one event. Do not propose changes to other events.
- The diff is the source of truth for what changed.
- The page may already have content beyond TBD. Preserve useful existing content
  unless the diff clearly adds, clarifies, revises, or invalidates it.
- Do not invent unsupported facts or speculate beyond what the diff supports.
- Use to_remove only when existing content is clearly wrong or superseded by the diff.

Section guidance:

1. Core Understanding
Write 2-4 sentences explaining what this event IS in the story world.
Focus on: what happened, what changed, what it means.
Not a scene recap — a distilled statement of significance.

2. Causal Context
Bullet points explaining what led to this event:
- Prior events, character states, or tensions that caused or enabled it.
- Only include causes grounded in the diff or events index.

3. Consequences & Ripple Effects
Bullet points explaining what follows:
- How it changes knowledge, relationships, risks, goals, or world state.
- Stay grounded. Do not speculate beyond the diff.

4. Participants & Roles
Bullet points listing who was involved and HOW:
- Specify role: initiator, witness, recipient, affected party, absent-but-relevant.
- Use canonical element names where possible.

5. Evidence & Grounding
Bullet points with manuscript citations:
- Exact short phrases from the diff. Chapter references.
- Makes the event auditable.

6. Open Threads
Bullet points listing unresolved questions or pressures.

Output rules:
- Return only the structured proposal.
- Be concrete, concise, and auditable.
- Use to_add for new content and to_remove only for content that is now wrong.
- If the page already has good content and the diff changes nothing relevant,
  set changed=false.
"""

EVENT_DETAIL_USER_TEMPLATE = """Current events.md index:
<events_md>
{events_md}
</events_md>

Event update target:
- UUID: {uuid}
- Summary: {summary}
- Delta action: {delta_action}
- Context: {update_context}

Current parsed event object:
<current_object>
{current_object_json}
</current_object>

Current raw evt_<uuid>.md markdown:
<current_markdown>
{current_raw_markdown}
</current_markdown>

Source manuscript diff:
<diff>
{diff_text}
</diff>

Task:
Propose updates for this one event detail page only.
Enrich with deeper understanding, causal context, and consequences.
Preserve existing content that is still accurate.
"""


class EventDetailAgent(BaseDetailPageHarness):
    """Event Detail Agent — enriches evt_<uuid>.md files one at a time."""

    def __init__(self, llm, accessor: BaseLayerAccessor):
        super().__init__(
            llm=llm,
            system_prompt=EVENT_DETAIL_SYSTEM_PROMPT,
            output_schema=EventFileUpdateProposal,
            accessor=accessor,
        )

    def parse_file(self, path: Path) -> ParsedEventFile:
        text = Path(path).read_text(encoding="utf-8")
        ident = extract_section(text, "## Identification", "## Core Understanding")
        uuid = when = chapters = summary = ""
        for raw in ident.splitlines():
            line = raw.strip()
            if line.startswith("- UUID:"):
                uuid = parse_bullet_field(line, "- UUID:")
            elif line.startswith("- When:"):
                when = parse_bullet_field(line, "- When:")
            elif line.startswith("- Chapters:"):
                chapters = parse_bullet_field(line, "- Chapters:")
            elif line.startswith("- Summary:"):
                summary = parse_bullet_field(line, "- Summary:")

        cu = extract_section(text, "## Core Understanding", "## Causal Context")
        return ParsedEventFile(
            uuid=uuid, when=when, chapters=chapters, summary=summary,
            core_understanding=cu if cu != "- TBD" else "",
            causal_context_lines=parse_bullet_lines(extract_section(text, "## Causal Context", "## Consequences & Ripple Effects")),
            consequences_lines=parse_bullet_lines(extract_section(text, "## Consequences & Ripple Effects", "## Participants & Roles")),
            participants_lines=parse_bullet_lines(extract_section(text, "## Participants & Roles", "## Evidence & Grounding")),
            evidence_lines=parse_bullet_lines(extract_section(text, "## Evidence & Grounding", "## Open Threads")),
            open_threads_lines=parse_bullet_lines(extract_section(text, "## Open Threads", None)),
        )

    def build_markdown(self, obj: ParsedEventFile) -> str:
        lines = [
            f"# {obj.summary}", "",
            "## Identification",
            f"- UUID: {obj.uuid}", f"- When: {obj.when}",
            f"- Chapters: {obj.chapters}", f"- Summary: {obj.summary}", "",
            "## Core Understanding",
            obj.core_understanding.strip() if obj.core_understanding.strip() else "- TBD", "",
            "## Causal Context",
        ]
        lines.extend(bullet_section(obj.causal_context_lines))
        lines.extend(["", "## Consequences & Ripple Effects"])
        lines.extend(bullet_section(obj.consequences_lines))
        lines.extend(["", "## Participants & Roles"])
        lines.extend(bullet_section(obj.participants_lines))
        lines.extend(["", "## Evidence & Grounding"])
        lines.extend(bullet_section(obj.evidence_lines))
        lines.extend(["", "## Open Threads"])
        lines.extend(bullet_section(obj.open_threads_lines))
        lines.append("")
        return "\n".join(lines)

    def apply_file_update(self, current_obj: ParsedEventFile, proposal: EventFileUpdateProposal) -> ParsedEventFile:
        if not proposal.changed:
            return current_obj.model_copy(deep=True)
        new = current_obj.model_copy(deep=True)
        if proposal.core_understanding_replacement and proposal.core_understanding_replacement.strip():
            new.core_understanding = proposal.core_understanding_replacement.strip()
        new.causal_context_lines = merge_section_lines(new.causal_context_lines, proposal.causal_context_to_add, proposal.causal_context_to_remove)
        new.consequences_lines = merge_section_lines(new.consequences_lines, proposal.consequences_to_add, proposal.consequences_to_remove)
        new.participants_lines = merge_section_lines(new.participants_lines, proposal.participants_to_add, proposal.participants_to_remove)
        new.evidence_lines = merge_section_lines(new.evidence_lines, proposal.evidence_to_add, proposal.evidence_to_remove)
        new.open_threads_lines = merge_section_lines(new.open_threads_lines, proposal.open_threads_to_add, proposal.open_threads_to_remove)
        return new

    def build_user_prompt(self, **ctx) -> str:
        target = ctx["target"]
        current_obj = ctx["current_obj"]
        return EVENT_DETAIL_USER_TEMPLATE.format(
            events_md=self.accessor.get_index_text(),
            uuid=target.uuid,
            summary=target.summary,
            delta_action=target.delta_action,
            update_context=target.update_context,
            current_object_json=current_obj.model_dump_json(indent=2),
            current_raw_markdown=ctx["current_raw_markdown"],
            diff_text=ctx["diff_text"],
        )

    def format_proposal(self, proposal) -> str:
        # proposal may be EventFileUpdateProposal or a dict
        p = proposal if isinstance(proposal, dict) else proposal.model_dump() if hasattr(proposal, "model_dump") else proposal
        target = self.current_target
        lines = [
            f"━━━ Event Detail Update: {target.uuid if target else '?'} ━━━",
            f"Changed: {p.get('changed', '?')}",
            f"Rationale: {p.get('rationale', '')}",
            "",
        ]
        if p.get("core_understanding_replacement"):
            lines.append(f"Core Understanding (replacement):")
            lines.append(f"  {p['core_understanding_replacement'][:300]}")
            lines.append("")
        for name, add_key, rm_key in [
            ("Causal Context", "causal_context_to_add", "causal_context_to_remove"),
            ("Consequences", "consequences_to_add", "consequences_to_remove"),
            ("Participants", "participants_to_add", "participants_to_remove"),
            ("Evidence", "evidence_to_add", "evidence_to_remove"),
            ("Open Threads", "open_threads_to_add", "open_threads_to_remove"),
        ]:
            adds = p.get(add_key, [])
            rms = p.get(rm_key, [])
            if adds or rms:
                lines.append(f"{name}:")
                for a in adds:
                    lines.append(f"  + {a}")
                for r in rms:
                    lines.append(f"  - {r}")
                lines.append("")
        lines.append(f"Approval: {p.get('approval_message', '')}")
        return "\n".join(lines)


print("✅ EventDetailAgent defined")

✅ EventDetailAgent defined


### Event Detail — Review Flow

1. **Build targets** — queue events for detail enrichment
2. **Call updater** — processes one event at a time
3. **Approve** — writes updated markdown to disk, advances
4. **Reject + Retry** — provide feedback, re-run updater

In [68]:
# ── Instantiate Event Detail Agent ────────────────────────────────

event_detail_agent = EventDetailAgent(llm=llm, accessor=events)

print("✅ EventDetailAgent ready")

✅ EventDetailAgent ready


In [40]:
# ══════════════════════════════════════════════════════════════════
# STEP 3: BUILD EVENT DETAIL TARGETS
# Run AFTER approving the event index (Step 2a).
# ══════════════════════════════════════════════════════════════════

targets = event_detail_agent.build_targets_from_accessor()
event_detail_agent.set_targets(targets)

print(f"✅ {len(targets)} event(s) queued for detail updates:")
for i, t in enumerate(targets):
    print(f"  [{i}] {t.uuid} — {t.summary[:60]}")

✅ 7 event(s) queued for detail updates:
  [0] evt_f72bc8fe0f29 — Mira receives a letter from her mother sealed with chapel wa
  [1] evt_f2e1969e1f2f — Mira and Arun discuss conflicting accounts of Elias's disapp
  [2] evt_18262658f796 — Mira and Arun find the chapel door open.
  [3] evt_c43da1c48907 — A cloth bundle with river stones, a cracked watch stopped at
  [4] evt_312eafedfd49 — Sister Celine appears and reveals the silver key closes some
  [5] evt_50b5adf351ae — Mira is carrying the silver key she found under the chapel f
  [6] evt_d23bb1179343 — A packet of letters bound in black thread dated July 3, 1988


In [104]:
targets

[DetailTarget(uuid='elt_45d617e4531b', summary='Mira', file='story/elements/elt_45d617e4531b.md', delta_action='update', update_context='Full detail enrichment pass'),
 DetailTarget(uuid='elt_b973f2e09131', summary='Arun', file='story/elements/elt_b973f2e09131.md', delta_action='update', update_context='Full detail enrichment pass'),
 DetailTarget(uuid='elt_9f0d7e04852f', summary='Elias', file='story/elements/elt_9f0d7e04852f.md', delta_action='update', update_context='Full detail enrichment pass'),
 DetailTarget(uuid='elt_03e8d4548117', summary='Saint Alder Chapel', file='story/elements/elt_03e8d4548117.md', delta_action='update', update_context='Full detail enrichment pass'),
 DetailTarget(uuid='elt_bd49df28a5e7', summary='Silver Key', file='story/elements/elt_bd49df28a5e7.md', delta_action='update', update_context='Full detail enrichment pass'),
 DetailTarget(uuid='elt_4f2c3df00e4e', summary='Toll House Ledger Page', file='story/elements/elt_4f2c3df00e4e.md', delta_action='update', 

In [103]:
# ══════════════════════════════════════════════════════════════════
# STEP 4: CALL EVENT DETAIL UPDATER (one at a time)
# ══════════════════════════════════════════════════════════════════

target = event_detail_agent.current_target
if target is None:
    print("✅ All event detail pages have been processed.")
else:
    print(f"Calling Event Detail Agent for {target.uuid}")
    print(f"  Summary: {target.summary[:80]}")
    print(f"  Attempt: {event_detail_agent.attempt + 1}")
    print()

    result = event_detail_agent.propose_page(diff_text=_diff_text)
    print(event_detail_agent.review_page())

    if result.get("patch"):
        print("\nPatch preview:")
        print(result["patch"][:1500])

✅ All event detail pages have been processed.


In [42]:
# ══════════════════════════════════════════════════════════════════
# STEP 5a: APPROVE EVENT DETAIL UPDATE
# ══════════════════════════════════════════════════════════════════

if event_detail_agent._current_result is None:
    print("⚠️  No proposal to approve. Run CALL DETAIL UPDATER first.")
else:
    msg = event_detail_agent.approve_page()
    print(msg)

    remaining = event_detail_agent.remaining
    if remaining > 0:
        print(f"\n→ {remaining} event(s) remaining. Re-run CALL DETAIL UPDATER.")
    else:
        print(f"\n✅ All event detail pages processed.")

✅ Written: story/events/evt_f72bc8fe0f29.md — Updated evt_f72bc8fe0f29 with manuscript-specific temporal/physical details and memory reference

→ 6 event(s) remaining. Re-run CALL DETAIL UPDATER.


In [ ]:
# ══════════════════════════════════════════════════════════════════
# STEP 5b: REJECT EVENT DETAIL UPDATE WITH FEEDBACK
# ══════════════════════════════════════════════════════════════════

# ✏️  EDIT YOUR FEEDBACK HERE:
DETAIL_REJECTION_FEEDBACK = """
Your feedback here...
"""

if event_detail_agent._current_result is None:
    print("⚠️  No proposal to reject. Run CALL DETAIL UPDATER first.")
else:
    feedback = DETAIL_REJECTION_FEEDBACK.strip()
    if not feedback or feedback == "Your feedback here...":
        print("⚠️  Please edit DETAIL_REJECTION_FEEDBACK above before running.")
    else:
        event_detail_agent.reject_page(feedback)
        print(f"✅ Feedback recorded.")
        print(f"\n→ Re-run CALL DETAIL UPDATER above.")